# Phase 18: CIFAR-100, TickNet-L Large v2 14 khối, Adam

Đối chứng optimizer với Phase 13 SGD trên cùng kiến trúc `1–2–5–5–1`:
giữ split 45.000/5.000, seed 42, RandAugment N=2/M=7, CutMix α=1,0
với p=0,5, batch 128, cosine 200 epochs. Chỉ thay SGD bằng Adam
(lr=0,001, β=(0,9; 0,999), weight decay=0,0005). Đây là run mới từ epoch 1,
không resume checkpoint SGD. Chọn best checkpoint bằng validation Top-1;
test chỉ một lần sau 200 epochs.

Notebook kèm mã nguồn 14 khối đã kiểm tra checksum. Bật GPU và Internet;
CIFAR-100 sẽ tự tải như Phase 13 và được kiểm MD5. Có thể gắn
`TickNets_L_LargeV2_Adam_Source.zip` nếu muốn dùng nguồn dạng Kaggle Dataset.
Chia phiên bằng `EPOCHS_PER_SESSION` và ZIP recovery cùng Phase 18.
Kết quả: `phase18_cifar100_l_large_v2_adam_lr0001_cutmix_a10_results.zip`
trong `/kaggle/working`.


In [1]:
from pathlib import Path

SOURCE_ZIP = None       # Exact path if several source copies are attached
SOURCE_ROOT = None      # Exact extracted project folder if needed
KAGGLE_INPUT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')
CIFAR10_SOURCE_ROOT = KAGGLE_INPUT
EPOCHS_PER_SESSION = None  # Example: 25 epochs per session
RESUME_ROOT = None         # Folder containing the named Phase 18 run


In [2]:
import hashlib, importlib.util, json, os, sys, zipfile
from pathlib import Path, PurePosixPath

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
SOURCE_PATHS = ['configs/final/cifar100_sgd_lr0175_wd0005_ra_cutmix.json', 'configs/final/cifar100_ticknet_l_large_v2_cutmix_a10_200.json', 'configs/final/cifar100_ticknet_l_large_v2_cutmix_a10_adam_lr0001.json', 'configs/final/cifar10_ticknet_l_large_v2_14blocks_adam_lr0001.json', 'configs/final/cifar10_ticknet_l_large_v2_14blocks_sgd_lr0175.json', 'download_cifar.py', 'models/cifar_data.py', 'models/cifar_experiment.py', 'models/cifar_training.py', 'models/cifar_training_large_v2.py', 'models/common.py', 'models/datasets.py', 'models/model_profile.py', 'models/reproducibility.py', 'models/SE_Attention.py', 'models/TickNet.py', 'models/ticknet_l.py', 'models/ticknet_l_large_v2.py', 'scripts/aggregate_grid_search.py', 'scripts/download_cifar.py', 'scripts/run_kaggle_cifar10_large_v2.py', 'scripts/run_kaggle_cutmix.py', 'scripts/run_kaggle_cutmix_large_v2_200.py', 'scripts/run_kaggle_large_v2_adam.py', 'scripts/verify_cifar_run.py', 'train_cifar.py', 'train_cifar10_large_v2.py', 'train_cifar_cutmix.py', 'train_cifar_cutmix_large_v2.py', 'train_cifar_randaugment.py']
EMBEDDED_SOURCE_B85 = (
    'c-ri}ON=DhmLQZsx5T)nQBM!hJxs$`S@Al}vphW9J-+Ex{(7<^Bde0m%1mWs)oT&YWo6-R9$_!t-Mr1*<5STwNJxxWpwWy5b}(XtSg{$wV#G+S7#8gKixpxd'
    'ShQflg4uxJeD2rmn!872Wxaa+qEwOYZg#)to_p^3-`9Wi&-}(e|A7B{-v9Rh`)~i<-}$E=Jixy_crr+*qvSN-93|6eyg5ilQPys?!u)g?j<Z&KZzsGMwpy*7'
    'FpI*$Vm?VOo4?4@>2=NX{7E{D$9~uI$9@BU52JaM$8&rl8{!w~Y@SS#@8S$!LDT$n97owSnVv$^^BCT=n$TT?8=l0|`C>v3AJe0YI5|C=hr@UfUD0D46upUO'
    '>EJAfXPs7y9-Kt;!C9C?fAFx~*{6r+(KsAs(O{mWQ|^l%=W#rQpW7Y!1rtmcIQ~&I&ST?IIF6^M^D}I3mmW{!d>&`%IXszXi@0`EdtC|UUNxBK$>7~IhUUX@'
    'l%2-mc}D;_jM}XbCUg(%HGusO{_M9tdIEsPzr(u%I~qn4MEO>${k}lQrmCP0&C?s|?j)W^Z3FJ=VTT_+HXg?_1baK)*#D6)5tS{i(VzXJ&;Q3ig!RF{&#TtR'
    'wM6ahlW{tDm)~`Penjij{$T5a+v2^J=kNY2f9s>)hNZ#3CwE#NePX=l>U@7B>qh}^C&l{T{=)zCJO2Q_;oo0^zhQbY9jDPSB(OARS08`0xyZB4lVrLXPtU#C'
    ')%+}-Zq<C>KS-zN@id92gV@W`bnabb(QFoH-YCtye2^uxdA?a{uL(74$z+yhb1%QjYojck$kI)_v?q#9qGVdDB_l5kr_m%1L$BBK{4m7lVd!^Zo*uT<j4zXU'
    'fKP*Zy)e)J`^DdU{+s{mg9rGBPYEG%#-|Xm0fq)6dkWuPzyM$t(R2tPi>5hDXOh#T4xT@I@@AvmqIdE~%M-|y)6{4_VK`dM7a0y!%psai(|Lqzn%Cs?q48ll'
    ';Tp4Oel|`{WbxPVlV8lQW`KF}`O{=DZ+I_~9R7VZ!{(xK!+X1!jbqj9bTOG-c~S07XY$cJ%?6^q*Uw+b#-2~2(^&L^1)7U_GKNKf>8Sn`JR0;JrfF0-PjVQ6'
    'EGxm#Fu+#h(#<TMW$AD+NKTS*GQW~l5UIoXJk9{CV3rMjaFJ&3;;dHt*gKj7h@x!hO`)-9oP0+R^nfPh@MgYjRciDNNX1u&PhNHzM8_jQE`Xf*(f4{@&}wc!'
    'Zf`eWvu*EpI`p}{yHi(f9KC(Ix{1#AUJI*swzqcZbGN-)ugNyn9mq~=ugwG5-Q$6@_UpQl^+vM0$0OL;<<Gs{wj2qx@Z{}LmoVu#pJxJlj;q^jEa|V0JzmoW'
    'RD`ta1t#npo&m*r!zybQE|iwVZ)&xdUw;)Iym)eSbZCt`na-WPw_D{-+DLJssoF3eDP)R6fwDwa0{o}zLDO|_<EQw^gk=EKe;N;=CoIr}=gBOnH^=EkoWUnx'
    '_Ba~EfxqE5Jl{7Sg#3US0pcawNv2+)4;^Eab?y)U%VMA&Q(1)`XHk;J-Z#;B5g%q*ngt{O>*>2G5KM-c+<x9Q4)dez=2LHQmZnhWq3GeE2b)FyXg!3H)oO!r'
    'l;_?7ogG1X@{4#d7sCKr_J-)w_zDPc{?3CEUOJ7jkx81xUj8SG2tH<bhQ+WGai2#>41lP3^LUy=!Qg59O_sphKk(j0SK|~`tpf*hy;-9^o@A%Fp7b<4OUF<)'
    'fQ7Ahy_bv036SA*q`H`=-T()Ie<LrB24~z`Q#HY-pKRgn8H^H^8Gn1T<Pr#eUDY&)-;^`vVIhu$VKPnTVHo7`c+`+%>mt$tCcqBmuGE8FWA$LMfocOnaizMK'
    '04;{|5sLy)<UBv~TKJQSa%;tsVmhCBZQZyN7kpkpz?(l=Bw37&OsDZFP|Nce8;eh4-7k{KS(+!-OTjnPRX;fIrmWJFmF?GJd_7Uygx(+;j}`Qi$!S*{F>eVm'
    'Xor~(EZ{-$=5&}$ypMX`7C{U_0uPfsB(j67*er#i2lX`p6JO2%wPEg_Y`{DC@@E1~jai>HybG8Qc9x$-v-o(U({GqR+kI7*b_GCY)`V@52LaZtH}E(0vDK;9'
    'R9QTyLfrarGM>hP3EaXIuDD6&4d?^sc~-XyT{;V07#&hmZ7wz#1I4=XHmpV4-1P9EYj}mGuR2zxPpnGoHF0V7ds$WKqEM-0Ra&dx;)HQUj<4F?D|mg`?p}8A'
    'd;kW#X02H$h*M8sAX2W7n5F<<*PHV+7~-z%;kwhe24MAhG7wMbYaO`F82~<v@&KOJi?kb_d5_Hg#Nl%a#|?9xnoA0J;~0k#okM(#J>rjOaf+fw*E>nmF^v7K'
    'VEjFgIseKRVUbcbYP|ha_sxS8*s(lTgkP~;e6q}ldmF@g4*RC-@v(4_rL$lQ06mL_fPJUEZNPng^<|nR-=))eG=2t)+3;S<^R^*Q8e)`bgb@%bnlLWD`&c%A'
    'Fq9{7DgvuG$1~aBX^wSk{hB;=CD_#R%tzrE*k`DEY~hXpj;aZ&uOUhgJwJ;_W1uR&U}`E`vf&X_7k`^FHC$3e!7@N7^b{z1@MqNOoAi|Nfat!365da}mQ8#F'
    'aGEH`O^1Qt3k0F+sZ+0KHB&F)ei^j_(J~jUE~eLH1z2telD_73-pY#c@Mv)Yh!qIAM)N)V91jc0Qz7U8CgX&TPdrV=$sn2QQ6O)^k`?eHEC^Py64+)JDIIRH'
    'EV=@52~`+#U@lOuJ4Ut^ncE={8T*;)t@jd69OK*|;z(j!kD9k%h~|;r@i+Um=m}48=qeCD3ygrCXipEroFcTF%;SkZ@`sm1afsBg{9vB<%)OfoZ!`{}4m58L'
    '>KGmws606h0c5fa&h%JT)TMZmc!<N1XaYc&t(#3kNfLzV9H12mL`QXV6BWQ97&Tl7dkb-kcIm1ELN2`tNZ3;Xhy=RCq+OyW6bVcy)y2^QM+UGqqXb0g91wIK'
    'iuZiHIIxVK!^s!Y))4G1KQ*;bT_N+ao&_Xyd6u9u%2;3nRn^zN+Eq{Ed7MpPE9CP8xx*`u1}VpdO{T^a5Y2#J!kq_WNro)>iy5f{BpJ~%fXv$hHrX6Ba?Q<_'
    'DN~gND69Nr0jI)AO!67<7Hv4rPNC^*v(Sw%IF4+Jt1S30rt$$orY=NA0B1N)L$!-ZYDM50kU`mX(Djb{#2pSacWA6g{iY__M?hSlztKrN#*#4<icuXI2>99Z'
    '2jV`Jts@i7wFDy#fRax>vlf2^C;gW970~@W8G?id!j&YCOj&901}|NN!m%DyU!E{9DFIPh?yyYu@SyH1l&pHMphRE<0yB??L2&>kh!oK)jdGw#39K^k{fiMC'
    '^JTh-!x^NDsU$TSd>^79r-}mqXChDCq9<Z0jAlOb%*v~l;}C^d2K*Ed56-1I)Q@r&(g~U<<>3Co2gV-CqY3O*-=2xwSn=hY)<poLtA=L~O>3!&HB^OeU##rM'
    '-Vy5haCL?z4e;Q=$aqL;HjJ|}8Z>bsgL-lG&BvOR(A64dVL=;D7Rgwsl_h&=u-zf*uZidO{*-jDjWsxO2Z>~H!%ZluL80GvFzeqtf6p{oFgIFTET8n1Lf0VC'
    'Vt9l#CS)5X{sgIL3MeEg$>V%(>%i~M+Ut$cD^c?$q`Z=n9}^`PxK1mLigp+gE^?rj9<_FPIzEqsLdoXj-7v|5S(L%r<UN8tuz!HRVfv1K3M7|P8H{tmgb_4^'
    'GLyJ)10X%Euk_+^9+xV#ibpdvYN$cWq7H8WlTC99nz~*?Vq8yrHUvHi=DuO0SApfHK4XOXow|I6rawve0e!-pI5q=_0WJ%0@L8bNqTb+Ha_xrNOg;I<<dE&L'
    ')s;MGBNc7eZh^4fYAlQqONUZe8k_ENXUIk}NXNOsPk!v7?FG>Nf`;Q=B=a+r<Sem+xhUeEO0rFEY?+IYIjZk-Y9xmg&jsXD&NVw<#gwWMN&dWH;(NndDK-rC'
    'G>27Ot+G_7o5z_oJV$rNcyF~@cj`<)WF?MT2O3ds(Q!lV6vvS^CL7)Ro@m-0txgj*O5f?Q!_{O{p3<9(H#3;pB%Y+%mE%<V<(!ey$Y~vu{7{^f;)gmcKS%I;'
    '+ORo}jba7Mr18CF$}Sq<c)$^du7QRIf@No(QCXgNO0AFQve}ej5AZF9+l^rWqiTl9Lb0Cu*05emB<q<!8rEi#=+v`zx}kRvJKkuxM2pm`;$fzx5bIDYJa!&#'
    'Tu4CaG@c@pYN(69kWU1!ph`3+(R4wO1lfa0l>unpq0ov(lAuh=^Z@nh_ViZ+IxLp-G^Q7}T+jHf3+SEK(_i&#F3<X^(X?k+<8>J|-=6wrK@3+@O;#Oj;^;-('
    'a;i@7<x*IepzD1CoSjvFDToacZYx(;0NMohGwFP$|62+1HK4vO<Zb9%5YvTBu4*~XiM@Vnlq*2thsC+I@S;<<Fw#0w$AZvnglU+-D!2{(hyTvs{_C%Q_rU}F'
    '`<IGtLugdY5+pb5Jal-T49Odc)W!330y1ut0rL|LOji|8*UP<NnrdO46f5y)gr>^+ZM}&G`MLah7Uk$nB!4EU{D%H;@_U*W{E(zi&f-K^dzAN3JeiG>aa`@G'
    'w9@scR(td6)mtfd!Gxf>Ff_!jrVz1?+x^<%>sJR~hR>cKzIb|s4lV3BhrSZFBOW}EYntc!*F2rI^)J?T<i5R)=nQ8R;Ht!miHbxfM74VD&EdiG*N5hC*qN}Q'
    'y$Bno<DhGp%1y9gc;uBwWI1~_EHBT7;oR9MI2pokC=*VSDSc%xLs#pDzhR@`XxI=QhB!dIiOD~OT@4#W7elFarlri!qR!54K=K0<^<i=vlbKB*V^b7il8urK'
    ';{sJ_rn7h&_}PhHN9;KRgd7`(11a{-7Sneq(xcWW7)O(nVbm2R0ga+z&~A0My+<B?fDCiu`vn~ukEpqr4N+}M?O8e`@%}8n<Zpq}tEY=>5QnkY-qPs+M~-s&'
    'F`@RRL^$c_2oQTD6p?`}!Q{iQH&4guN#H-iV)Z&J91%_==U>lYK4aq5041qzy&2^pGIy8Ic@d(T&=MTNd3-q!V4x{kE&auOw6SmcCH*}6IjuMRTW?Z1cqmHM'
    '&FK@|X9#ne>W$!Lt~6K&>12}NB5R`29Gu02cOf9kY%vdx{nLbSCyUQFV2kI3d0!qrdFlgOJ-8V5aC#10t@<yY1Ch>pMt@Hazj^ufix)`6FXyb1kxp7HQeO_@'
    '+1v|W9TDH$FvofjjmPouHUFa5b;sls6C)7i+7}qX3*dLf+QH0VwjtMpr~E~ForQcN5W+XSiwyDDz$8f<X+Nl^f-J71hN6vv(s_)8H_FhFL{e_3pX2vnmc^sw'
    'vPVsmD(Vx@Z~6$65kB{u^T`Z;LT5drdu}|1wH>38b|i#w3l(@m0nI@T!BBZaZHa;Eb;HxMCI9a6m1V_%Iy4HJX>K%YZ)_;Gn2wX_JEMdEI>S7~b58*N93?2x'
    'fW0J4Q@mLhHWJN-cM{fZ4&h+S1FB25Srj!o4G%_~UVvyLbW(K$CvL8F@wv213*)A}#GX#;kDghiJc46Y40xv61=a&dQg{aN1SU*W6~9EYCx8mXG$v{D2=>wR'
    '^!ck`8eaf_<DT?q<kd#=Nq*i$<JQl3`MGifRx=(C5yA4Fd77J*8OCQZai!%KS$dHNa%AhT77w6#8qaC&0h|OyAU8Z?HWDyWOa(MNjk8&XARiVT1HOc9PTDwj'
    'CRCPtV)%fsJ%J<hESV`iT{1mS-yvhi9G2;{Ae8{ik4gK;LNy(lh~&5w^{y-BIkXbZg*J~Qdg4*=RD}_iUyAXYCnZ_H=~$c2;M8G*xTrzeEp~A$zEOHE_MCze'
    '0Kwhuy5)#c9)M5=7LRHkgftR5x>J^(LU)4XkDZ;dA6CU|`DJqmoYde9_|d^^0spdF7@vf~Z48b>HEC|J?@-;cy5=nmPH1a{u|*i`q)K;FVvuT|1qC>u&SRh}'
    'AWWQttT0Lj#O;gShb3s_I2fKnVM^>v7v+O=I!{g)=|aGd{?$aej9*-Wy&W6e7Q4XVnOefQavLn&ohR{y=&L!1fS2$;_v`h-87esJu4m#j@}crnEi+^{#%|U8'
    'm{5L-WtAPUiqo~PxqMT0iI^!QWkWT=b{Tc_o2)3?GGvvh_Z{#W0ds$XuNI~<l0snMeq9#N)f5KNjAXiOdRj37pJQzR5W>#@5oL}*t}~ZD!}ucY28z}f`Q93{'
    'PbY#ePX;Upy;hVREQXOPia+e~Asl%Dfy^k8<Y9CkCFAI13=J~mj{Aj%l?;zdek|5NE6<G4S1bt3a@o0BY|*`#w19f)1myd2RX`qvs=O*R8oh90aTwOQTnhU4'
    'ST2<?lIu0e)vQE8UN%uqoYdBOh;qDSeMQ*CV_KcQ#Uzn8kolHeK8)32e^oZhui-HL_0uQ9+tp*gguxj)b_~5dL2`tCADS&Oya<dQ^Ag6`15`vzDqz&2aTU3#'
    '1n3)b_67>|ZGV?i1C{PsCThJD@eD51DOYvF%ZM1$^(P7W%6h{^M#q1gFD9nPg$rK|Tq(>(@+fd5g)I2z0ZEchiswGafODV5;vi%CrH?aetI3N6$0{I66Mgw|'
    'O>ddO0(k}~HWIdg#XBesi#LgZOB~1zLQEg2NjP$oVlcH{q6rKmsYu{dXwFD9JVEUn5KZxulyRa>QQ2MwYIuQ>XW=A*VwbW_stFPf&H+_p^qhfi$^j&z=n2$O'
    'FxOL)<Gog$`UsgtPO+j~jD%Rd2}0ISbtNQ@%kK<g=u4HL79YjRnVJ;GAjwNYEg&u&!TN@qzL8i${K8eJdSThrY>O94KairQQKTjWV45xZS(6lmD4PdHITz^a'
    '%d`-63rDVhBsg-TVzlIp@i+?P!V^K&YmXEIjlPA##bpH)x~WrDn(i`7-dBQLXi$2RxiL6C-nJac!f|A31jnV``*I6+i%w_u@)1%|U`;}2H}qqoVYw_$NRDB}'
    'Su*mUh*2qmNQyi<P6XE9_!e8JW>nC_|9Ah-%Rl&E_8vUIzuznB;idjuXyFluHcrqR50xNUJV-IXwBSj)k;h|(BU<4X?AfGL^taW$n|k#LD#XS2**Kb$r}RqA'
    'IjhOd9~@k}M6>?-`3o6dtIf2saU(9GVR{ZViZQoV3)(Uv+X*c7ccZMBXqdFMVY0QWv>?WD*04GHMnO-}FmZBKvjz*-D5`E771Xg&I^pU9E-KN01jv&&2VXva'
    'dwB5n>o<qto5OFOA3cBdQmXh^_i1QtOgQO5Jd&CtF+AQ9LWjhgii51RLRohI_RW*$FAv}7o|#Z16-8}~>W*m=zhF|NX&jMUdQTn1S(wF)ZA|KleJmqg2dYO;'
    'H0H_1kc$GA5kLyN1tfz{X=woRto=tu5{t$c(%C!C6fqDj0NE7_uyGRQ$zUT|0H2-t#d)wrrcRD^R8Y9pxfja;hzK?uAi#ie*Mf%pwI&qImh?mf(es_FU~7|^'
    '!U|kDF1+lsX9tjR2Lde{;Sm+|P<UU(&<k7^TacC5XvHvjK3zd;$HZ}s%T8^w`W)$RHvE)6r};cvz-cgyQLhsdw4`}78qG;0gMYIHIpHWBnaR8rc@L9)(1SXb'
    'nBSK+088y@H)1gzW*efoY~+L$b!G`-OAO%km17T<ElCj$#F5Z7nnleOEV1*j67<sMY7Kg0&&=kU@R!yD$Abk(8c}u%#E*S_l(Z#&$}hB`8eX=TGBewdM<!~P'
    'vy@1)DUPg5x@7yxbQ)qfWe&3$F2<PuL3#lhrxgvjsr>-81dUe9`5CgFITsQ?>cR*8X_BLQ0`PFS7yu-J2ORL;B;(en1>f{&e44_MaE6&bqGYT>zQ$L$yo+hR'
    'n9azz*EGg)5)Iw~V&^Oc10yh5jGMes=c6qQa3vC7QDZu-Dl5csDb@4OfTZbei-;l3SQY~!2y`;_**m*RO|#?}CEDr3`MjiUDr3BoF2TYDqF1(>@=>KFBVqNT'
    'HdsRTNUsS$C4v>I6dZ~QZ5ft3vjzhIk(-68LM_7w0Yuqo2HSvjtvv!jLvFb+osO@h<Ff2%%>#c4s9k&jIxz+o9N41-pKy8d9JMNG6M$9Gj2%NdOtPGdPS%PF'
    'PR>(dcsPhNwxmvhBaHOyOiKLF_dfBYX)%=BMpz*1vPzziV>>2?_BFIFBmYoy?JP%l!vrlqA`wNUmWpXgaTJZh5QO7Uwf{XMzKEY-%qO=UwAo7s4~g4~zR~W%'
    'q*8i0Rax~!x{6?ME;B=mDb$ZiT$YMl)YLOxhau2Ll5<1U3LBDxS&X09g^N%&HOfXoXmpk!(!m>$IILHtCM@N#C`Ysh5ryeQ6O9QR0fB|JX13g#P+VdKv<B$v'
    'l|^y!%F1iwWgv*vit*6azY)CXAej;L-^Ewm0tUk)*2usY+%T4~WY#~{Za)~YRz?<U_cArS+untd2s1GS6jJYHU4ZCKHcWgdP3EH>S6*uqzG12@K%@nGXz9kc'
    'qTQ^He1Xz}zvdsS^wP2~F1`81>#t>ppTgmcItC@_vxyB!KpyETekuG871Ltjj1$9HP8g}wVk;$-K~><9WD)DD_3F9@;0vWbM2(p{sh4ud6eT)VC)P_jW#DTY'
    '<5^7n5UNLf%!dLoy0;i8RluIipijm+kZy+Baq(m^gwff<N4tcZo5odljT7c^oavvq5ShI=Mk{>^Y{iIm4w`K@oNc0$@NL#cw6A4XQu$dEi>6lr!W$Jmz*CLT'
    'b2SbqBqGSti(^2Zex1nz{H&Znaf4ZSx6H4x%5Bm1I=s`Y(Oe8&QM|wgPfRMc?bJXljr0Hpm+nO-+Sn6L8UTi_N3hWYh7ud<2~DSO7?o+Op$p4!Es0C43@r;B'
    'b(9?rSWevvx-<>uq~6pItsRBvBqwH~WQh(I>q{X^+kCHY>O#GO6D_o?0q~xU6U-cq3s~M_^+{M2Y%VT$M2f^A2u!D046F~SYm5L;Mf9vBBgP^r87|aDCBH9*'
    '9p%y}Qbh=EF?7|Fq=(VmP(F})m%FkyFv*ax2q7sQLPh3GQ*%YB&+andBWOhsQxrSnVJwfQb7O>Ms|*V)%tUJ2j22kbWQ<y|6dLXSbW2h+C_7)<B1Dy>!6D_U'
    'L+PAX%Czt-8OMk8&Qd{bh*k`>yUJP?0rMW`XQFlVLIk`9O#2kUmd6<eHlcbAp{6uPz7;uK7MmPH)k-ucx8)rE$bZG7;zm?<JvnAISfnwcN$*;Be&g32JRXgt'
    'ptX@^uev?nO0$fvfcwcTW^}YH5r+R12tYDbxi$F+h=fEHlM8^UAJo?CTm_2B07NAiaZH$J3Q7$#!h~K_1>6+=v!cZ?)kjgxHM9}#U^15{F~*lEbyCBMV`15>'
    'Dx9bApAp~U6e(0-H7O64jlCG8<}#XCacf2`8N8;0<QVkgSG^R9Rv7|=nSgjT&5%(U)?RkIDyw74l1nJE6S>3}J{PRE4SFqYjk1Mn_NY>xUHLIT=-*U8R+e*#'
    'Bf?~+JwcGSvtF`f#Bvd{XaF}#z+#R|3Ro*oUFiiV9Y`r13VfnTnc}!@-Mk@|46@jC0jwl<fe{I3y0`#0uHR_iH^9=RNpk#ejO07HB-KL^lA+IGVnKL#oTCl3'
    'MR<f6I%G4kP0xz;#tlPML0E5K%b@1i@ND;r=eOvfVDJ&dsJbVkUDL_o5|a}5Y;`vK*fc!M0XN8it75*>7?fB9-Ndnvyb^3NkVq%tH~}6cH`xP+>7S;7F$yTN'
    'Vl|JedBN^i9-Pb3gq%a$UwF@6yn3C>Tow|(?7M>*3k@x7wHA<zQ;KcF-^bjQRKnuhEZ%9Q^GBH-APT<HDa9&2h>)*a>UL6g0a?TG#N=103(HkU+1PrHV^pzC'
    '4$X-0U7!|M@~#%wU~oZYrbqkV7-q5zPLRoDF-Il7K_(3@XU$j<VJlyZQAsXmCvo054rK1jhG!Z8%^X#Q=5&OW?F7|xaL%UCF2LQZ27T4ZD5zP_W;SRlJxwgC'
    'YSUcl_z?pro-D`GCAnYOgiwTpmbJF$I_*jp-KsV<Y7|<nv<+9aWU=pIG{AT$;C-~2VTDm)xeeCgizN`Dl=6-r3qeO}#T$kY*IRB9l-P-b7Sx+UMc7q<wZ7A7'
    '^8ktU;dGv*v#S^IRhpipmjr8aQ8~`xG?Xw+YYjxu3`$JGmzmY6W$o&DBsgB>Fj-guaX>(=CV4*OcsyWcaFN$d*ssNgVLfWYd-O=ViWEIGdL=8WW6_=7483Ys'
    'iX{iC(}Tj*r_i$Y#Aid?6d|nzJ?RS7w4bBZt?ICL^g=o=Bz++rjcBsXd*=D;Du85WG)`xE2m`o?vY}Oj-nci<>B?HL0ow87hF3oJ3z->=vkjf9kyZ@o{tS|7'
    '%$6PR-J=7hVZ1;-HJyn(YBnew?-3FfqpfJ2s{`xOh2FJ~nN=}d1C_33-d)QWcHHXM*NA`f^Z9UfTkU>bI4(&IrOE~<yfKtLnD&=Y()7IcQhUn#aY2eWEp0)Q'
    'Fn248s4Z>9tx2;^Totci*3)*2zK&opB3n!TVh(GP2KjRR_1b<h<z=eS*c%D3o)}yZkQUO*OZEqyir^XNIYdGm!;$t*1WbuU_gTC^>uoY`Rw*lGzdg%J;!@0f'
    'AZAxFVIPRuRq;d=VjBwanR_){tEGy$OAC5jdfvw@2WnKpx@^qprl`PNml3VnrK`M~w0pSWjiy~Q*7aX=1n;fwa7MrCg8rm4h1YJ~vGQ&q3{!sHLjA1n2GBdC'
    'VYZ5vEIq_AGE15g(P0;!X6a&<0|kb2Ff%qCGZ83HDJL$)=vx{g#ZDPazwE=xwS!3EZqQUJm723eNz+FTVH2o!k9}n#GM)*;hWAr1sK8L8US$cVc|?$~YhOlk'
    '2=JHBfvzJ8ry=ozV*c@q`v7!owd+TZ)C^_(gY5q1#&KBW^LS>knSvV2NZIlPidUD7GdpWaS0aamO!e7qnjwv966A$^D@F<VXj$9Ja@F-pOK#dZt=FRK<*n)g'
    '88y|AZt8BYvAd!|#aP_zK-bv8X808b!MYCqu7wCj`Kn%vXj%4QU}|r8GIS+QRW_=5i>@<@^~N~9*}v5rB^l6aILy(P66nS_J>8@|$3z-)qhk6x!$%lZjOfTE'
    'h<x*xaw?E^pF)O599H686E<{PTf>{pYtyL$pLj3c^sdKQx4C<IgGnM2V4CG7c!;?Au-|QVMmIk}jZp7eqoeiGM`2}=pPAWZD_EA#ERIDUFZsJ<MyLEb2ih!4'
    'Pcuw~BE#N{Xt!AeywlKMuXpyF64R~7fT3cjhM~F((Ww*?i3xNmqo=iLYsC?8Q&LM5I3sdK`#$-iYLSCmtP(c9NA27S%^Z(oO`Tk|kGYc&t*M;5jT)|Adq0)b'
    '(r{L@;NA9(6(w56_@%hK6!UM7<&t_G$YB7>VJ^gt_^HmJ8(s+SZ93a%_pTWjO9|8uDq`4Cfq(iR&;O{i|KI`seN+_{sBVbm@WAJJIwszgbMeWbC1k|kCWCh`'
    '<N3x5?*&%z&O0Vgpi^=*Yc;%DF3pltoURf0G$G4599-YU98E~#4$7yxZQW$vtlh^M-p3i*afS+A?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF'
    '?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6+0}k&44(|gF'
    '?*k6+0}k&44(|gF?*k6+0}k&44(|gF?*k6k4><ha|Iw#^f9tP3cz}PuZ3G+waR*6ic4fxZHK)^B?c<L>u6_KIpLl2<J{Ok^E~b3-B)zQBuJMj8lKd=C3dOHd'
    'JaYxlh~X1LBNXKEc+~JNO#>Q3;?jGh^^vj**d%#!I!TjZaM=Kce>RQ=u_W`{@s}U0=PVvCc7sb0Bet5YLhpdkyUms!gBav{jN+~%z{U~%^#TR@tLO&hMZJEg'
    'u*8=A>M^{DUwj?d;8xetJ6hUZ(bVpmZRNO><+gZ*tE-(=)cbN(MS)vOMlTn3z44z_HmESfB|Nju>t#BBjwA=2+2f%wem&70T;H(!sxIR<Q9T^Z%>}RPEjvl_'
    '0iGIwh}KqpHjaRgo4(hUX<`&&aM?r?bkG9!d!wy(<k8{R(`3G+`7lI3$uQ*h%LgEuQl@C+FTcwFWD&>T#UW~m(yZ6sT~bCvrN{X0C>vi6@L)5G$~81*oiAo_'
    '7Sx++;D%CCbSB44Dmw9XbF(nsn&}WaYR=MhY+7NS3_&(X&f_QNr?25vN3q&u=x^8U+NkVqV|Vy@*$yyxO!VAS5JVnFKQ0co;hiK=Ua)Xd;~F5?MS;GQ0qhuB'
    'r($U3F<3*q&CojKp_SHChy%;98dpuNNN`y%zo3}}x#NuiDd&5F#fqBNd*f8f<(4|Ye^Eu7t-Y;01L!t;yB=Q4v0G+BoXapMZ6kAi&2CEtt)*USxOx|7Q)KZ8'
    '^UJRd<XB^UFU`d;*B4*7X<2bKbT<Slb~B-_jD8fLuuStZf`+q@Z9=>%G=L^6KpNgD%De4m%b?i=@o{t&XTWl!iYjBujU797lf8LR`I#j{1Q-Lz4396KY_7Fj'
    'YaOjMw7RdYxqxDn{9(VEMMKidx9rC>8~mrPL9unY^(zR9ty_bly%s3K;YCoM{2eE}#%zCpnX+!aq9<cfH5_&>cYb_u*ts<vI(LPGMMX}=>ENABMZ{qOZ$HPE'
    '@4=!irSe|&RwzQp#lATpV}i%=W|)kbd2^f;5`3;#8hDPEi#!fbrd3V}CFb%L$11lBM-P7~zcj@P&*1LOh>Lpdhdis7uc(RcOBZ(u@|zo=a;4Qm%P-|3V!8RF'
    'f^lH-X&BawW$6|IpW(%c<tz0$df>l=Lmi-tEX*j5P@!+b$lbbLNk=c%qE}1w@5S#do2sgGGOemoHU7&oOKUcvc_@>Q35h@j%PGk!QlgJ}morlJ&R@RuSM2@D'
    '%`YaZ32f~STW{Tcw~Rwxw$Cb9tfd=A?V7@K0Vn<_yfc7gp|8obd~g)_^R>)dcUb2353rPbmwWe1dAp_DyTej${Xn~Nc)_drWIPU*t=5VKTfHi-g{df4h0bLb'
    '(Ea}Q>%Gomf1o|<=6{yT0V`RfJ0OKER{bjD#4m02cfPmPU$wr=vEw~f-=qjX{L=qQ?c4T;UK8h*{54vXyKJY<2VNB0!2UP>&h+Xx{)Gn*@b7oDfn9oLXcK!1'
    '-;D&g^C&})k(wj3ZBxbLE_v4+rQ>0oEiJN`C*!=P%yjc;3L@n2X?mIm&hp>TZxMj2vsTil49n%V&?y(wGnj39LG&*2o=>9F_+>oDCw1xBLM;Sm^ZBg1xhZ<h'
    'Mb&0JTx>?eWPTOx?{1paH|y-vM*qA$OLEaUe0hEK7ALT2OvBS@RIO^U_btZRe~V#*-ei%_W&A#;SB)+xkTLhZO|a{{2@CTrxwe0ckr(`%@#5kv8Jtmgp~?}c'
    'Q>lIZ=7krXq~~!H`@AB5J>XqablQ^<<|6bFvja|Gq=RTYzVc44FxwiXT>e%CrsL-PwrMRuO~<LBMM>|yYYS=pMK(qcn_B~0Prq>d`jg=fw%9&7X^%S5&g0!E'
    '-fQjbjUGQ9?hW^L_IF#YRwo`6t07oLwlTlD@%VC%sR-g>D9fWuPvFzoa=&3XOx-lT?>Va#u9$K>JUMwhYVYpvY{!G$-JRC%$<C<t_+)2)D{k%X>@J&fW!qIV'
    'CI>RCg=sM<rxE$EnGe`0Sw1O)rmiu?VIbsm6V!+ZHGuEa9DNG%9=r6Z2l6+&S`{BFJZ14rgwOPr(5WVjCt8f9Pu?=+DGsxz#wy3BznY^|-yB0NJ}4+<3z;0$'
    'OS=?73$=mxSuTS)@&qg#8b0}|5cF8k|4x%XS;TA@r38G+UQFRD<<~{+GVLvmOIGc|yJA%a5muQmX5*MmmIyTNE}0JF%MeLip`pP3$(g+DXBe&F8eZK7;$KIf'
    'z+^Cq=V$3qAuf$w?FSwK`n{+QHLxXl#Q)x4Hd_oFhhh{r$K*NkaJ%r+h|aD_M2U~+41m)_;A+!oNTl028YHi;n$FI|!J7|~*;O-1b>v)x{;2XXp_w)GqL%cS'
    'Vr7q8oXt%I&O)7IO%nY6akt^M`h9zYS7AuOOl=PP1&oLDKhasrD5Fj>4LObsqY=U}<#8uGJe?;bLTD4}RKchkHJ2=JPw-03U!=*Dk&@{_fkqlpT*d7(H&ES;'
    'oQ1L3+gCRy$E<e+K3!u|^a^{VqW!!HY-^x@m+(XEOv<Eel*{Xdoh1DNR}514ir8awqsr4bOU~m0vS?g6g_i=1o@~jD24W_1RZn2Exu{5!^1D#%H*AMm#muUS'
    '(u%2l$Tbx+tRU!38gprp?QL8n)l2p;(mC3dHt>yoEGXK;T#!NJaGTyYQHDXE7iU0ijT7%%ebRl4%=Nd#ePXc0w{*ULTd}!E#t~_6AKun-v-L`xi*10XiBn8B'
    'h9!4^o2;2>wpTT6aY;*B-rw18eK5*o4a?Y?<$icSS5i31_j>Ccv8B`1pv`#<=M!c%Di%WDRRcD)A{rOU7nw@fHkix7a6AL-u0&NPVkm}Dxo<b-XCB9vdFjYR'
    ')`k$U5_PC(4CN_ECa?bOlH*1$C9`#biw8NTk@V{>RAye$G`x4qI3@c~<#~Jt^m(AF9P?SC-*UGFPPMvsSF^faAiz`RQ^mp-z*t;Z0FPMIrK`!A8K;ZM3GmOb'
    'hAiBbxT)`%+=wY4K3DUVYBGxTUNW1fq73mmLMr2_#RCws^9Yj;^Q9oHB1oq5l(h!1S8G~PE7B3U$Bp||v?5tzA;sHEJ6ep#CSj|h5M23K%bNWP>4L)xB_M1x'
    's?-)KnI#07I)J~P><?w^J`0gOV>pU3mdDlbK#W$;QKasduT&@uS1d!#ks=0jQA?nRtO8XqDvQ`6+i>{>Es7YTOET{}vRQG+kblU<QFHU~`sN`(l|Gf1z6&C1'
    'ko7Vs_|6~J3pUdkAi9MUa5!XmNkEQex=zh4yl9F^AhZxRdadjZ%O|TbGtBJCbGkqle~icCl?EP6<6SbF#cJ1JWK#94VXC&y^qb%PxBsm#f9t^m{QF(4Gvy#Z'
    'kzkJRK0BbZn_ZhWUnJ8w%Dk_h9IyfsgW`?sV8T7do59J)>1NcGBp{9fJn6O^>sB%5m+-3;=P(4ScJS)!mv5iH{31Mf_41oT`1<N4avE*y8oqQm&)lbCTtq&^'
    '*zg97PdSIDhYD=3{V5Gm4$iL~{_(+!ub&=14PU)JeDegGK2q8|A8orHs<Am04aS90Q&yT8e5}>cR0`eL0S$$w$p^_b87`n#tEr<ySagacxxny~d=PORj}l9I'
    'lk}WkLL>4D9A~b-WGW*R=$0-s?Vss$3$T98)oCfxg-G0pyG7k?CR3A;5&$oi;XIEjY5Hg!o#yiLx)}#a5xXHTN1}c`%3mT>!oc!!GM_Zt7{}-F*n2ikXM{&_'
    'hN%RbHCjCnj-;5Cxl>Q2|Jh(>QRfwE5b2r3QBLXH#T_qrAK!Fv#kb)J5~blHvd((}rbX;@wwET4G^&wu`Xu(YItT=)D3aHyG);ggQ_N<<k+M2_Qy_?mV5_9>'
    '+IKOm4Q74cvRBRcUY&)AXWCPpG-VL&5HnGl&+#|+Ps@dw0XPV$D}s@ldOcxY{ESo_Gz}536o@Eb@`cZJ5+lHw;W!o#42cO7kOl<_58s@n>AQd$5fja2+;91A'
    '2P31IsG<BQom3x2Iv{Ns7jGH{FO2=+hiq=Ddym9#!<>am9@pb`e=X9ix>vnRS46AuBpP5Lo;0TDg}sZzkqy}56oFeO#}L|Uao&h~Md+FZDv4@Zl9j`y39KYm'
    'pG8!kQdsd~00^4&xUK_Ptzu%>qRi4xaH3ERKTtJ9>BhQ1ogdMoau7SBhRm$T6e6TbW{#|Rc2!J*h4j0iO=9qH*oH!6d^ijhI<J9kfkHRJJU&C}4%8EyKrR!n'
    'e-@u6cr%Jjz7`Vdn>)kcF_^a?vXD$EW%YpRdU2p4$EHY|8Zb$*XV0U-iHmXBE$~Aa9-b&>A{H@;0VZ4t?9BN35GgRl>VOvPDJ~dGt+Bc_tw7*wWuD48??$m='
    'Fii3pz*d}N$Z3h5$}8fAjuW>;8>DTTr*llcjc<ZdzSTlaytZI5;l<Y3^l$+b;_>dSSKUPOh7%3yJi17Dk%!?^u5f*UI6&1fh|A%k;vk~X^|7%*-UA!-QP9xm'
    'Jv72czVkMo;%U-T6>BoT-6@?E*EcG2Q~^|7Gs_yu@qW{1@j<|uSyrivZ2I-%ZpX|?M|~XA2VPUezbOSaeQkKHy7!4^FIufCFQdmxJX2H}hdk3H#XRB>LuO5R'
    '5kPpF$&BTK-W#|=gwc&ERxX0tGJ4IpeJ5h=PBGbag21S?oSqv6UHKooo`Y*Q>bO!pidi^Va&t0{eAVu$;4MPCVXM``KavXM*uBrZwtRtqM(eXHqE4Yf0z)}<'
    'n;;mK2-Q?E#zD+vMQe<sqc2>cm=PlWPyXM3^Y8!D4<6v(U(v!>7Bhd4oFstxS1f)Vokh&`n)g29T>$8=W-;O^UHT(+KO6Zh9wZ|uERv}kl2@wSS~37pI;kzW'
    '$eE=y7DkHKT;lI8s$}Zk#!n4ZS(-QF>3Nc&2S7X*X6L{^`1<n~PmaQ$y?XOUN3Wk893qxHd;SHAKi%!t<K6DQyjhn<MK>M<`WDS8Y`ww@?<Zc+dGu(jQ#UW#'
    'oJ7+Frqr^l<6Tu%w@enpNyW{Ws+kZ)`Qm+^g<COC;9NVKMA^I57iijx+4E5QW|T%pD62@Q@A(Ww0IpsSzWgEJk1%mAh<HgL>#4IC<WdQyx`t$F1&qqhg|Ob+'
    'i2m>Wv!Bks{*4C@@b6#Lh<<b!KEZSdl%&#5fa1EST!sM=Vor-sjpQ9z;8`){!uOwS;eZ8>FEB~b8bRTwm@!GmN7B`<lrWoUV6VMf^29JYV0D!l*XDL%n2cvA'
    'nogTXa5^Dmfnl~zZV8S8L$7$=pfF79CyG{uahKbn1Sab>?`W^kqs^JsOJ>&63YNhlGbWrPoARs)pW%#CJb0JD_=~mez}kwhTlKmMUv<AO17D0c1mH@O04zmd'
    'bZ#cojYu`{OeNScYS){Ksi*~=8b7u$f$I!2j7PaE!V7a2V8c2U$cI77qr);T+Bj&SEzHu~CChuOuM=JurV9mOvuOuJuxNo0{n!8HU-kdaUw-fa|NfFDL~oP9'
    'yO;4?)1WL?<OMJLs1=D#iTEHSYOKU;az-a95?LX44I`aI!x8h*u%fd>dnTEbtv-^}?M?Xl>1*MP?2ajxiZQb&F{E<VR;dpjfy^8079qQ@FN{<x<yfcQinymG'
    'k+oI|gBAa+Y*~0OS4K%y`gL<eu{GtV7h=X$Awl-gEm!P%g>&2Mi_Z54#x-NGZq>}~I+X#YC1iQ?fskcY{M?dg>%tiMAxW(GrQkw1HS-x6zwBb>XjsEoQSj&x'
    'Uc*^nuM5!iwU{7byy$tC<p5{cmq2w(tEb*Sy;%Ah@UfT<Mhw^$0xe<I^V%fNm0{H@1Ew4)jrrW<tZdJgN=!6>ATcbE&+6-Spo6fPXbZx-f?gje_kNXU7F8{h'
    'YO8_XhH@{zu!8F1-dTp|x=>}dh?UeoUw){R0)WJOibDxHoo%+|R3wg|tej-5Lt7nZD!N^Q2)hN5EW&<R+O%Xg>d8?S5=5ZfcY-!jBbR}}pc~FAACuhfmHRPs'
    '0oX%vWla$Mp+lokgNr-sPS9q#I-i2E7<FrO2s9$EwbdOeOyT77?x^!#WgelP105w@Eh3>kNY=vC9GFEbRBYAu4<UU{f`)barfT9XdZ-)BI47+DP67Brlj&Y$'
    'OPdZ|EpQ~*6Mb=aWDmljUM?IaLLUmv_7P=77j$tRCp~VjJa(x8Yay|f>l@`(AREOT{|%c1U2kEn`4-ui`m(esxEIaJYob0)$(VtJk(dJgEYJbY4Qt^Ex<s}*'
    '+iv#C*{^OyY=PBGNg-=s#ZVfKz`*(%13De2C(stkza|}6mHn_@v6rfbXY9p+l6ML@bJTN`5p|BeO1M<qyHw1@W2T6!ua!COV^4z0VOZ2AavP|(urMmqik%+F'
    'v6t-}cJHEOf|_+ectqP!G$c~kIkw42nq|m$rP6WH-5c>3(R8l5q2vZ$%Ob3+7qbffYb4B(uyt~Ux(9c58<lvK?gfJ$Nv$apCF2PAt~SYfxi9p5m`|ed_@TUb'
    '({!RK9~&^Vt&UPD%O-jc=|ZxH-J)uU{3eduo&A2}cy}BA;jf*YUHEsW-4OvLc3V826>T+COATnJVLomwZ+?u<r?+puQEc4#Q1?jC?A8jL96E2HTaMH<yO>LL'
    'IhMgY;^5jK!T=M<|MwkveT_a8zqqWIj>%{(S<Lb{I)!n#-A0T$o$}ZP9(xY#Ty@7>1>@dZi<ftUqqhc&Z{tgEWglYxy=wm7=JLg7(U9=l|LITu*8lMP4<6v('
    'U)K6NaaUkCW?Pv6<Hif`yxpZ>WK4b^4?SR1I5!{Pm^Dc*$)q-$pOGIPI|8AX8^_xWWfTY9B|J;>1bvGiDgkcebHom`zM(;rtmgBaFO9+<AbO+1vswGf-475&'
    'yvmZ?m;opeo-$Arj|VS^vOiAFyY0uVy=Lq2&f}eWv-S)RT959ZqYIt01f2vuc{?88c+9s!V^hO)kZ<a8`6m0w(Ub<zGhcz6OO4_ysF%gVr^#SmTf;Tea0i(x'
    'tz^jr<5Gwk8qlJ0)Yg91YDaIMd~p~aeEH<%%flDsoY@9q;{m1d+S|CR_xBqfkea$E`SkGh+b_ACvB}!_`tU2Woz8ac%fly6%}0B?`!(nNkrde?E~6Z8oVWej'
    'hcqp%VOxS#e3d1qgbf16gp(~I0tK@hj*2222}PR-QCeS@k&`X>`?CyG3)O$_tWvt2^^#my+n}mVQ0wSJms_dc%R0q-`@>kMKCp@ELzt%4-`FnGl$<f01W)Bb'
    ';M&<wX0!2?H%vz4AvgDOOr!xDTAms66rII>K^K3C;R;>&(%oE?wi~j+^?;|Y2IfJaPEu}`pHLi7xiz<swVJzbfPAfo37%qbmc>(;egwzGsqD=!s?X6MtSafh'
    'IhKMvoPjiUMl6^D`wR3rIxf-#k-pW@S;KQaEazaYyxiJKnxpdMa3vgnr>dr~cR2eu6A>4cOrmMv2LxDDiGC44QwEKx6fB$J%lT5>cb2TNUj}E3>ARfLz_X8^'
    'b8^FEVp@4vF+q$puV_Hcu)iKZX>iLp=kJnPV6DD*m)HB2j44`0uW&$hL?ZE|&L#&^oe7_yNIKh<2WL_M^U?;D+SNGV6;}#ODW`C9Py-RmA~(8ZWfAZjKQ%Z?'
    '_(4ay`~hsiCtiLA?ABm0UrHEOrg3k0XUP!wiW0uBNjg6EUL@~gk62}3o#}EtW3a;inqvJTpb;179RCaEO_8G=69k%t+dGz<gv&#|o%MEwb#{o>*#adm!Kg@P'
    'ixVgtV@plAlN$3TiI-XRbTNSeR+1t?<(?C-IKN_d)}no_U|Mt97p0`TY$2Q%0~D>IYo4yQ9Zt^8sV<t4ZL<7)cu`Tj<6u|>q|u6L81)K9yce!*dxYnhA#HwY'
    'aOMIUo3wdh&cI<!nRunh!Bb2#o+Qi9b5ldG!_i+QmDL(&x^4yiVpkpBW^oxIMN%=y%EeQ<tmILeAVwO8UWVx=5ZPo71b}Ypj73JVp68U!9J(P(^egvjX>4@Y'
    'D%QymgA1I2X+bL-lO=Y2nel6@;aTFSzAB+G){fa~ze)#hwN(x`H*8uM@!s$XG=tDk4xw%vpGs5{k5gMF$GK?9Teg!IBWHxt#RhMgS)pP)P<_QTjnElb0R$Qq'
    'gv}OHFf~cT+|_FC?UZVM?7c}BK;%RSIUX>0i{NB2o+lz1!oHWMLO-M!N8(Hbi`py)1`+wjXr1UXXz$|5IyY${NIyK{PrU6qvU>aQWWQdK(4R1Qt5Gk`>@%+;'
    '5E;;Dr(Bx>eLP~Doe9rRy!H)V9Pi>hY89%u<#e2`#Ws2{jEwFBmqFw>6Wy3ydFb$Fg-Nnq&xDVfrLAmWuoSoZv1y>|FXt#Mby3}!Uf^S93e*bn5_uv{1cO`Y'
    'SSXzHoXDwJ8XC>2?U+k7h5Fc|<@DMGT7<aJ20G+edCTphK$a{^Bq`qX#Xf~e^pKW|dG`=+8$i4Ibw~OOYkHkKO|Nsu>CqIdNueUKiuji2h|0#QIlAY!b*K4l'
    '-En?68+*;R_52+3*|z5cxY{V5V{%B_jv;l{0r~c_F*)#v)~GzF)u^<6Cs6O)6_rZ!B2{!XtSjc_9MtX|flWiG9F%L*;xXk0wj0PhcLMU>9f4e)&yGQaI=7ln'
    'CGspi0jNI*<u|FmY{+!25Kt<+RkZ7sj$t-b%^1OzJ`>gm9rtP5L7ru2hFiHYEOTJ^p<EYaKa^I%@>r;Ue~t;J6lk~~6nzzB5ElCvC`)9pj)W$<(4wU2mB7X6'
    '-=Z1xH~;uA{KNmC_TT~j{l3Et8j@i@Jnt0kp)c^cci!oG?QPQ<irB%1#fvk{69NMhk-f+uL%{|!!y7`Kh$@1kLm|CVR*)p;Y+-qfFrH&R4=u-0B4#g@Mk_rY'
    'QZ>wT(<p_BKx&IAWhjY<P48#PaDJ9!AlTqcj=%k>N8#I^5Tet?2_~jc?LGu%op>#to?-Y5!_k5y%TSslfp(;GjzX=iXB>5>s(CL{Cs$VMr|Op|BpF3*Tjo~B'
    'WhdQjff(EY0*Zbc9ucy=-F>H_6khGn|3ICJR#9|#TgNO)2)A+G@k{y<TC#6CE;Y%sbdk+tOH(4U1OT?=m=TI`QivNfv=L?${-;)YZ6!&&)iO5W`cA?3T9QA<'
    'T9SL^$j-eU<6e((&&A$zv2HF_F_OVrXK8_-RY(7tZ0(~z{Imb!pZvuK5Ae@}C;1>tX7hYAIz7!|a?m)1{Tk*mo_@gEe*DqqBF{EYlIdnVJ@;l;^Rslih0NX)'
    'StIwpfGXY*RY4=pVvH;%rTAd(Vx!FoLgbH@;akTxm~|LO`7Aqy#mQClaDJsyZ2Tfmh1VSBf*dC&vYpqM;(&R|s~I^F@^d@}8XmeRG`v?L<DP5`y=*W)VeZX_'
    'wHoxo$u|Mb<7^(Zh_VIPa1e41<*?p_$){sX{DZk{aB|0OlJskbfBYI4hQp^JetGl!tHYOXk@e$q-KUjmx5E5%7>=`6yCqW_7GLZXUqnMp(~`AXt#<XR)>h$N'
    'bx+k@E$ge)Szn;K#wvjpLAIFY;V{X%v?elHT{y_kyJ`iF@l{{sDXX$8i;9zBRXyB2h-Q@G)WEmi(pE>KWB_Ef>zx1{mgz+rn2j~PK%y`&y`=mfvtbjj40?th'
    'fxJ%%+r;i_o<rxOtB}AI0th7jiRbV{>}_-{HPQjf@xv1*1_3#AO%dthOUf6DC#<VP%?!aHrb`skR1L8m`jrdjZBp5yyAowxg?B#sO^jnoGlJXDuVda9+!v0+'
    '!iHvPDe%UkWV(o#3jrhls{}|bY-UUqJ{A4P25ISf*Er-GLn0WAPC?9$fhWPNODAZO>KUs*vuP+S=*cgp-I|t2r4Y0)yE#;S>#Ty}Dk*pXoge!mXOQ1-E@mVV'
    '74FMHWS)S$lF16GdZ4R#{);#AJ>_P?GZa~7zh7wXXLOKyPvb#!C2R2E8V>1MvBKvF69BXKWH9i8pVXn!Xq-m#0D*Pvp9AMIPiJkvkE2k94`)6l!x5$8Svokw'
    'B6izvvH06`w$T<Xi<LO`xygno@&Zl{^X(Xa7YF(ko-p3Cwpnn3&%zNm^_q@w3rxi%UBxyzS8DJ@UDi6KtWz@QpjlodTpW%7fy~`6QWEAW6OuvU3LaMYOdJ%='
    '$vfC%fjBk9eMRKu9;WYvyO+xQIcm;RtQ+WNL|S^&Lrr%*8J2pU4DZnKSuE}l^1VOxL>{>z-oiazOr|EsoJ=_k!~YnOBN=kCS>T75n6bt|oCuhb%5?+g$UhJ}'
    'dgxuFx0r<M%_jYn1gm~yF%!UsU@zX;6+APNm7QFn5l{?;5*IgJz4|yDm>dWPVz!^)Z1q!P>i)*YhEGH^q*wqA2~nwEfXF8Zl2S91Xchn~fw{&n>ar>|i=JBC'
    'Z~}9nI|A~+Smy?j7<QlPJ68B2gL>f{bLjd%pUT94I27vU=cqr3fpC(4q`#PtHdIzHW-^B(agI+Ss7VJM?D4Kl)0-1ANPevOBs*Q;BD|(IAU5;qN8yu-S#<G='
    'O`Z7{DpD|IB-%nC;|a6@Na}MGS&tb9k?M?o1wNMXfg+v7<5|zgXXtr=Pj$k2FA#OQ^rF7JR|ew-?5$<}mI6Qw?m5gClS!0aHL(pfwpW~S^x)_lI$_ru#ANv1'
    'V}K@#L-;Dn-eLaxHO4^)_J%mHeRkf%Wz7MufMGx-M181)J6|+IAJ~u*q{<8-zHLe-qd^TcO}b`2m<)}xnMsw!O7y6(^~l2K7NB#2Dojr5i&_Xj3=z4*(C>=$'
    'l8EUAq3I9)tAFF4{NvVx2l)3_453Ng9zcv88v>s2b%z*?B=14YguCB7hLxFLu_hj#DvLl_K2p~dyr*z<7Ud`hm9r7%XP9AD{mRD<aY|m=DSj@pF*MbTk^3q<'
    'L;njnxZhj$YItwyCY0LKCvTq|9li~}I(+*CzjvwlG0{o=4gTpPzkh8>BXD*bxjxNH3aJ3VcZ0=Pyiufi`kim)r-VHO+a)xvSj@Ydn}g^)hHVVzZU($*N_Q;2'
    '%aZRlSGPC<bVMaUvB`^zi(-p^W6dVTb&Z3a{m~YPaQmaZ_WtN`XRp(aMq4M3cjCS6t?kE=Z{-7(cRe5>H9<i+F5(<|6XkCTvfXNMegJrW2!O7JXngXxwX;9i'
    'injJUqtRfm-P(<}TdmGuyA$mX_jlHUsO(0A^QKn&*n1`DmcCbFgRTo8VgOVc#1XQ;kwMUPVF?k3c54%QXx0w@`0bk~Bn}>Yd3f+gM_+$+v|=CVgeGB|hWq&O'
    'AQ}zYJDtw<b~M`C*&6L_?F{3+-TlFEZ^yjf)U4aVy2I_0lW2SEaSY-<;M-`}iXNYgM)Bjl!AW!y4=U<zVcnhW_+-$6N?XxMd(_(6e++HKySqCb=y-3ZT~T)%'
    '>+WuC58M0O?VYWi&Tu%2_FFr<dna)`+=iuUwWEr<w6@#(u#D~P;dXnpvps-jPCA`<=VTj(yf@eyoS1d#k}axBHCtOd?cvTo^zj&G+XmolZ$+cgemiP+_V;45'
    'rr1qQ;5s9k?P&CPumk85$6LGY?a>z09|3?PK&*ZEZ^@qHTg2!p+TB+B@o+ER8jf~Go$b!f-e5Q!KHhqK0$)e_LnkAPbHK`@Ry^80X#t2kTb<p}R=hVF0q9P4'
    '_u6~WU^H-6mc!qBytj4I-hmOpG4^<{H{5QGPPRMSyIcFCA<V#M>?R)3*bt1eAu-&Pn;o<E0==d+!OK}#`$G&Rs#1i;gsx5nG*-pDO|N(m9Ak6+<HGi{LcCEe'
    'Vc%_ox2%BVH^~E*$+@R`Y2f9uV&GiC7&RVScn0+aGc4+8%?+OI`YIcx#{qWUv2}S;<uu?z1tS2nsTwRQVOTNgR=3EZNsYzGzn0D1faJ_76hlK&j;jaQa7+NI'
    'SQfy#0ktGWNhzhfDEU#(p-x(FJYPNC(ZITv-)<V-DNOU4+rP1e6xpCuInL)1Y(e13fTKcr4=)#_OL`2<Q}^epHP)+2N<?ix2YE}iNtkKIqD$EVFaDZP;(NSp'
    'G)ot!XWkzHuS?g9PTw&bCsR!6!8G~utDZHQ(v7J`1(ogrSKVwVA@(^a8zmGvqBFL_PCQ#!t*Gw~r7UQR4D|pFb7Bp39T+B?zKF)}On5d#ggs5OX!959aO{Uf'
    '716w*CpR%!m4=?g0B!U!26f!+>)BN54Mi(-=csJm>e7HaN!q%D!o>^-K8dl`RA5&cjVx=Np!hz`iWQ|w0@=+S$;1m9ktj}4c;x#)syUfj=@ESpz60KO#kDi}'
    '8IfKgFEp`1?v<QHRNH(}m!F;R)m$SzA9iZusK74JEV^7W&||0<FG|23n~T|ybA4fap*c(O{w%&6CZ}+e1(I41pk0_TNGxcJCFKUqg#?FVc-%ZXdU3#S8g#no'
    '&3r(0Jt*J}WC>&^aVsNBi8_PDl+?iq;^D{}rxZ_wS4)zt>y0DyBY4Y9CdqW!BpxQ%ES6Fm-MU$2FsCpm7@HWqydGT5ptx<JMnmT+k1q%DZ0;S>2QY=MmP66B'
    '9{-BgIdM7zRSB}R(75EGWV7*Byu7OqJViQ9R8uAfz#GFJgXM)ehSmrR^Vp3xZmEcbGjbNHiy_T+XxRAz`0UHTZ_Xz(Unm{M5i~+#t1&hOOgX}uqJl7E+^{t9'
    'ntfN?$UlD6_b^rPWJU={B;?5616&(aGVKj?$u*l7V3FvWES_00h^&M&>*EgMsnG}7eAdPY;~tkOQ2<+_Z?dtP-v)(uU%q|&`sho#@gxgG(>)lkBflI}DYyNG'
    'Ke5?{P`Bp2Yyay!&NiN$vJu7qD*Y}QkE6|<W(#1)R_}b{;Q7W=rCzD~H!E&hRD_Ht!c+`~0mz(8;&d_Z?Y4-Z!6`9Ir>I&r@JtZB3qkbI*97-TRmv~C^G2nr'
    'oCmVWrkEk!p5ha@rt=Npd`{<QK4t-{JKIxQBs@i1HFt_2Nn~vrkM7~^uRa&y*fnR}0@+rBT~57=8GPvqLsB=Jq<qD6KIu^u_=*a2mxKc8PrX)mX`u0JfMmhw'
    'Wz(#VaA`GLOIw(ngrpTMv=W$y=c)$SbGB!8eVY~}t@(L|zkqN`TnCB+G)9cD+iZ_+JXqgNdPIGxXTdc<f`0v^PHO&P4{aNx@ghGfn01Q(xi4h<UO{xb30921'
    'R^4(PQGXgER~mcK2y^sk`*h9aRb-uDwTg7&ct;7mFNi`0rEt$$Nk&+_3CuFua7`)6d|nNCmjxWeB@p4*aU0JXI)Azj5emDo#VC5>JHTjCzAUDM6M=Eg)@xO('
    'dHhM=J6h0X!g!VIm5Cjw-j>G@tgv}*7j5(fX2mFKk5zC*fvJ~ne*NYJ0jJ>!R#5zU%XD7+dUYheXlN2?hc?Ra*X#o<Jbb#)$Ap=<t+!$&&60~LicLW^MDhFa'
    'M}6QHfWQv<2t)3bkNe)WKK5bJuVHL1I4ZHIR&iI_$bT{(YfdI&?gqGf?p5Mde$(_`04zMgN?mKf5FVMz6r5pr`dvUi)aP-5S>z(h3#h;wnw*w#>F~AL79$Pv'
    '>(@qeH_fKBC6;d&Hml?1!rjyF9QO+kV@xb3^R5_eJ7m^LjCX?P%!cBzE*=!`eWRyz%^AP1xa2%A``#3VX`(KH*-!)sY)bMli^mcAk-}kMwpK?M8S>T<jV3v('
    'iaa$ucti?>%G>oUy7Tokk47T?<UAHlAmP?yHZjq4QL{0;h6GGce3Hrmh<1;(xY9IfP?<xivg^9g!qyKNYbbGair3o;Z!Dp#T%x8#6-PFR3oc>R=bNQ%a{8tb'
    '!rNx7uF~wFx<Fj1Te5=HmafUWDIUsVLhSqW5$p-?;4Dq>0(RK3n2f2|YV%!ug_r88!vM9vFjBYbRhW4h=48RJ6}DK}ybfCkuu{b;RGQ@)MTKsS!y6Ci%46~w'
    '<7>ImA03l+(Sd>rW>Y0b3uRt#@6<*cD_nO7c>n9a|9ATgJ?U?CQ@#|OI#FXmS2KH#Ci=?@2$k=b@(shApcm+(<ckhM(Ij1Sjb?d5$-r*Aej}kHbc};&%IZgD'
    '9Tz?}thrlENqzi|3UAWoha4t<aR&c9M;*o#5GKj7defn&bo*JG1%!gncx2x%VW=wI$72@%LV7wDw0f-lMk>2PwcBuGQaQ>)=+jXr$8C3AM>@ZAXjWa?Y<*U&'
    'FgaB4hqCk%?oR;7)SFYAc=A7Ncn{gVM>bS<aRN&nEHztY;@WNe!%A=#L}01aqk6{dqR<?qv#U-}vVk;;L`A=<gtBcJT9+`aaUMRA1_E|C75h%aS9-%mh71f&'
    '#J06mG11>h4P@kOSE!oA@>jIC7SWL#N&#gR-LV8BJ_K&DGxe!Z-ju2d$y5N+$R3>gvsj_Z0JhYORcvI10Ea>wPX#>&`nTGx#jarA%Bk8`FY4eSbnvk6HhkTB'
    'OPj~_9sE8kH)U~34XdY0SXv8UR#+drtGA$p?X5Oph4g}hGik$*guV#*Or80NMl53wh_8MhN+ZE1XiW)uzYnS-+PV$RAv57GDoo4eIAH}hW{2qBpCZ&uh;@HH'
    '^@VFJAE&Z2nXUuHU}RAkOHJ>lz9Mp_e^OsDvQL$GE$N%E-3uO)?x4!D#cW<v3<xr_LUOaVu%dKa>$bb8WM^1q{;~J9ZSLtAT^L>jteY{_3_`z>W~FeDm7J!%'
    'z?Sa9&3yRRtd{us^ku5lPI}1&jx#!tWt>^$YhxrJH~G%S{X+^Pki55IpVb-WMe>~p{UG9O<a$u!@T*48M}Ryo;T%v{gSr@gYq7alL@=!7M(Z4m#EzRKcuE8n'
    'GPUffc3XOQXZq)$cS^p$HNmT`ONF#<GQ@P+YjZ$`LwuB-no(=YL~4oocXXK7<U%S=uM8rF4uQgbc=O;Xl_762z_1n0^jLcX7htJJIcZ&_yZBS(KtE3@!Mz@W'
    '_Nd24_0?VDVxVUNDI|IUh`s1et~?eSh6`L>k4h91!x5Vbvvdm}E$^4;sM~2>w)u=mZ5AsXD2c};m$5j$v;(FZWvEu3@Z9MW)2j$)BxD>|R)yhm+s{*p1Sjzb'
    '{SX$pq$WUn@LKRy4)}<yGMZT_UwZ6buO1ZijQaT++@fLE=indLqEhZ>)hA9GeeiL4*=~JozUy0hzaEt29iUz8wG9~B=@k@(Pql^-I2bjTn?kX-tbdg_t&9^n'
    'oF0;(;8qiQ+C*9;45`GaP~=hu$rE}+4pN}64fE*Z=avS=7$>kKte8Soy%<V?9>C)0vkplDtE_13f;fES-!>is<DOxm=K%*s6eA=4=l|<B|NG0|d+-4N{=kTg'
    '$QIL(^-ZCSN4E1N!6ELwK8r8`#I_ECP?>4k+Z%j|ayhROO^vJWU^r$ni{Cd&g5t*Aff0+7S(c(j%Ml;(T{5Hih{~u3`;M2|n^&*iN=@1FnDMG`b8G=wQR<O1'
    'QZ*UtjLT}|Bw>>zgAnJ&E_`8r7Ik)Zt;*#r1hTq2%Z(M)?93UUx-Ulh)NUIQU8^0wesw@8lsc_eO~*ZB1fs;M@&v;6HmC4Vq3nA*;Q&a+Vje;}@yxFsyn6ZU'
    '`4`X(tq27#Pe;jVzDYSgHgDH9hdUP?sW3?uAZk#uC6s(ob*`q@Sv-m}y@%Dd0Ggw?P2lw@mHSj1?2x3M;4!7V&y?#Z?cNw>k1VUkA)IqLcnKqV&0|_qa63xY'
    '<|437&DK#9_JN26OtL1E1Bj)b;Z|;MQ}{KK4Q&?e7GvNM(oGz59`F*9n+iubmI4{;R&D8NNO<4#^^*Dx&zb0{=iC79Gd)_I;?5ISMGmrb7LLZrOkkbwt|aG8'
    'v{~%Up0Q8WLU92X8Lha$#ue(^+YYyPw!+<=7CKQFG0nZb{rzya)ec+xBEt}G1|v*9v}5GOB9%h%lvKzK^i+hdiIRoJ1uD!y(!|6(nu=8=iYQ`-7>u2$K$8kn'
    'GLZvEn4;;rIh(t)6gUJD5F^!Tr4#8PTPB7MY=dZC@)s&0LC@-WNuGq75x$ES=QUMy`8>u8lzoX%3R4_p3$+`8FGH`TGLG{mAphlKlr^mNm9K<4SwO?5dgE#Z'
    'Ky~I1b1Bhjh=%`(=Zi4eC4NH{^(eG;Ni}B@kU--|qh3H|;SWzDXV1F^d0ZF1*}SH6*|fCF$v2}s#4L1|LH!2ANAg%H8AZrefw{%FQZvJJP=Wu*CqGU*l7o$m'
    '_=w$FQKQ42tv4cAb()Mghs8B7zmHib8=fKfnrt``KYV7AzY^p)djkr|eEvF0Xfax1hYOcx(IRJoIyUoS7?3Zwj8VHIR5>sa#6YNb@m1GkU^w<f+`MzuP!P?5'
    'e~5?V2cOGWn)=O6&2qA+z#(AeeC=PHW)Voc(!03eSPVslbFuO>5sYB-Ur%PwaN$%NF!c*cyQl7y>8Y~MUx!Z*pS^hU_V6jWp$Jj0TeGhgS~bD0%O)dyG1=7x'
    '+A#K|SpZLF#+y*05)-8cAGO29WR?f=73Y|k&U>A@W!x<ji1J95Zh62bGgAKc&2VtLZ^pXYYgzDP6QZ(akSiK4K&9SLJI*@Oq@*VSAr3X%;Y$v*5@%RE;V^@d'
    'i>#@nP(@Bs$`-z5>aQS<yvait9x(2Nnj9tar<^%xN%AEm=o717yN)*m7HYuGG{mYehEccW*OiM4zi$E&$>uq3pn%HwOz+5}iLZfzhe!;g^C%f3KT|%WEnToL'
    'a(^iO>ld%T_M*W6o@TJvnbe&~X;`0eWnh?!Xm%(m7qJqRmW>ZwV*_ez$Ql*7O7+obC<_df#6*#^fN>&NQCRFvEx2NGg`iq4X}ItSFnw?_>>&r*V1CDpz>u_b'
    'CC628h!jc*g+0sMN+d2D#p~rF;)}8&sDL^{SmbyJE?63<Wr3)0O_A)QYGXO~xP*&*BbY0x=>y6OO?#rbuCzZCjcbO@b)Lns7>EqSQQRi3D=OZW34OPbB|wm4'
    'Ze7T5EOKcfu|fcJG$YcOa1IwNE=JBx5jF57*`R0BbVKQOSqn%PtA@0tR2&2>jA&`)q8L28ymk{CcT`6jKMOJLOGtCG1w7-FS$T+wgNoMx+C$SlsWe!66k?Gm'
    '5zxj4<F*A7Ya;T}Z52!CKu`ykJ=eL=1fC&{g*EUbb8ie%sXK3OMGeh4e<t|EZq+>Vc{<x5FZ>OL2+w$uOasATTJ3$}7(iVB!!lhKXq@PE8R-&c$<Kk~aRk%d'
    '0-x9#P${%p1le9IQ^*5euwlxHw`x)9udg2}Bfb!|9+r9Whi}SDpFc#`F+G$UM|_Cja9rcHQbXOr#1)3abP$H`o%%^KJ7@%fLw%D|LA}2btk^Ca#nwsJbh`ns'
    'Xj9PYN+VN6<AU{D+H{#gku3vuEMFI*T&2M*2fC*7!m1}3!&RB1q@Ksv7LOjCLkUVAMfp|llGSPBZ~eW0{2%?^g9rHc`xUl}#e9-nYKsMuKb@rM4N#jWtoj3@'
    'b+DL!1tL07HVWw@|24^o3h@4D?G!&I%ft_6UHIn7o99nB5AU@fwJ=LetGNUGM?|@E+RYYz@8I`Nvvsqk)u3gX9WI8gR%<8BB8f@2u@z{;fN}<~-Ma}16O2O_'
    'XeKBMg{tEnm!zCPCkak6^nuI+x3UlPRh*E-3fi^-Z096Gc0}glP`L>(x|v_fun4(uYlTCIZK>M77T*d{A|rpD^$T_FYR$Ddm4>&b9-9T1DzMb6b=@i-%=e?^'
    '{*fxTN35oL9I90+F39(z+I~M{lgr>_+lDesD$Y;D3}A<t4>6B>I?H>V#uEQfU3?OqPLug!829#oGh>iMV}MmpZjxgGpni3Oq8^40L?_8OnP2sg5UwhJ8^e`x'
    'lGfLo*{xbPDpD8qqOIcYcB@|rV+MJ*tu|wjFWrycfbrOAzaRsKz-4$WzBw!i<}QGOYwUyfGiMO&?+qFEhKyeqL&kFc=FVn~Uz$O}^3htC7rQrO{J6{*?nCo-'
    'MvNjW{mV07(5G#<DCT|qbu?Kl<sz4vE4bbU^9205#1}YQT;b?eY1nWzb#LeRRkw3=TCKYnIXZZk*Pjy$hvd7x@88t1M$^9zHV-M|=r9>csv4L^Z;@p2&=~E#'
    '<>cPH;WTgj!|vbwKV@h}eVKVfyNLp!(zXt|m708BA4?@FYnJf7Q;xe}p>av9#(iMMuUue82|Z-JBJce{;yIi_7t0j#QM+}!s19cjc4L;6MofT<{<(=!DC?ok'
    '&TgTQUQZc~wzS&&s-Ewzc3vlF;sljPA4IeLy=k>?745OJy%RoeIio%H_IJYV)_t_cGCLTbU_b6qk1cHttE6S|?0b#*D3QVYh>ssZ#K%^(#bnKpkAl&JeIM@Q'
    'K<>>a_vVuyZ5+sb1joJU<33p9*UIwo9uXS_6NnE-yzgv)%NY_GmIk5BzYqMlH;CK^e%uFs+y{Q#n=d}R`Qo=`XMeBt*B(5;zmMJK3#IaihS9_}TsUkJPoNOt'
    'aFiv$#pg?|-@Z3n{7RcG3el}q1`HIoWm);|j<yh^sj9ZtGLXDEJb3>4aMe}LO7F90-A8SxR@O?7g}VmrDy>%A&W~lZD%i>HT~cHj?F(&!al<mQwlUqCT^*8a'
    'lt)P%3$x;L&z0lG=+3NZ+C69<)!CVRtw4v8I$@G=_c-d(;hm8)X`M^c)q?G>S}-R>&4pTG09$|AR@uOAzj7}5q;mQ8oqFp6I*PF@Tgs(ItjblDh7tFw3fqWz'
    'ce{#LB9$`I1O{p#RIVJeBn%X67w>suCm&m>EZt^8DerOJW4U<!`j63aC<ke+MgJC-DcOLrDizgj!<DSZi`Ca&xJDjPMJ-FwILE##-I|Y?dL33c*_Xt1(VKlS'
    'UJ>?gqiLmqbX9bXHk^K__?+@AtDF3A*3l0T_oFU;;svU<I+`1J?MS`lTU=WxStBOetQ+t)2!H>3v{*8K`vVy)<@}1W>77lQ5;$~H^vPlwE-_h#x3F2>#ft49'
    '^YvdLQ>P2n)-ZaCY7cd_hy5SF`BR-J^t*4_LdCcB4Wa35o=lSOn887o#5WD0omT6;t)Py4dnPC7T)KPv=)HaP*Tp`%{80Wi&YC9}U6BMqa33pjAmoeLjCO;k'
    'z<Mpd8G8SFb8gX`yNbvAzKy*qmp3qEMV*wHtB)|%m9Vc~B@5MD*L#<hRfZvNsrmV~w&qn#?=R2l>>63Y{#-fuUo%Ve8j@ns^jxLW_)!Gexwnd`Hz*h~oyPO!'
    '*7AGn^?f+skKb;s<fHH3fW1c3zYdmcoiED@-qX>2^97705R69s-YR~d4)K>`aR2f@|CjrJ@pm3Pz`wt07~G47OE$EBgO~b6us0*Mn}#?lFFJ+BfvHaBxi?Nv'
    '^M-kdF%8Dzc(d9tp&Qxq+nU=!`gK{_C#Zg^ww<qI<2JM>4XHj<nigQY&f?q-3qvwK6H=$vuF*+bRxZ+=fRfB;!E>fiMeAR63{IhYt2DM+uqBDBum!^<)wnAS'
    'Ng_a}qA}EfgrdmWO_Ih(HNl&U6&KBP7Eepn8s0R%7z2OV^L<kEoB_6s4X*zz$>|oto;QYX0W{To3K;Yz#{1s_E?rmLz!<s^HRvW(2FS70<@NEJy<<FnGR5$)'
    'u8b~*7d87>lX}bW7U7`XD7W*8*RC()Y)r-mnjZqW$LlI%A;HS7#J)da?dBx9l%prbRBs$jPKJ?}r59Z<7{U6$kkX4||2!Ipz^b&7we~g&FUDz(rn0(jTJd{0'
    'b9|sDF99dl8MunT*L#<1`UFd(C2;{QW9@OdeBUm=3{G!x>ISj6h?slJNr?dPI=S-Bne6Dxjh75qXhvzoMYSlRB#jChB%bLu1H+x@4U^F*&iFER&uU5=Bp6d%'
    'W>A33Prco+1rj*?F%-G9W%a~jJ+Cdx;oq`>9$rQ;mc<YmLgXCdVLYsWlVnS*y&)gc=CF#h_S`DWkNV5T^A$DWC8!Mr(a46G5DDGGBZ@<Ept*|XI72u;p<Osa'
    '`^QuqKc!iB<!Vat57_bZcp5=7%Li8=<qqZZId*}hqfk^Tpi9aD@fxR#nrECJwXv2;fCKZniA6>mLQa4!0a7lS4m=GvG)oUaOaS&qh<7%T#&Qc}e1Khm#ke%d'
    'wI<GRuoWr3i(GRPs97K~d(PDLoJ>LKMW1PHdV#-@lxG$h@0(<tTOtU-bnu?#=~aL)pnBoyIf!*FtE0Nvk!Wc-wI&+lDOiX~S~JM^ZIjOAxYhR_VfWZaNIlXy'
    'F&)%h@FlD^8K3uouhGsrM<R|eX-GiCfC{LCjRT_@11?fkuiuI@gU%LyOGl1+rc0XlfWr!^63qC8XdJo3!xk1Uk=L6CE$=gFM}j2;h$5><4~-$<i^4{#UO=NB'
    'bcv!Cb~cVqfB`f#WqLu6x%>XHuGvR*K%2_qSxSb|W8;vWq7@j8qQL2f<K$gDPR`PFs8|UGwgF;=*NCh*@d?{MN68fJHrNSW>W*I&D$@^Whlj8h_L308j5Z=2'
    'RTY!w$2b!ZS_0Hm!)HTo4ar(&c;~Y!gRhR0KK##kWMR4@lCou3G&<6Zy@FTKGd<8Ji^PbWnFLweEFi|g1;meK(|@hk9Py{PhrKz-65|{wWgeYoaa=yDt4|ik'
    'P?sLXNXwZnr;}ukFd&G`Vq*PhQDmE9gSG{@J3vw_kZ%$@2)w7Y7<J%0@j)h6B{L{o-L+1bMQ%{DPA%n;1_c8+c9D~8c$ioGkwSt^IZ)W{zzIZyEDc9kY-BOM'
    'Lf<VAdH@&~2XY~yufFaKRUyc=j(hERqupuX?WOwXstwMXLLM5xldW6kk!KG?>@0;+f+x1njGy6tTW4<-H`KC;i>4d=c7*FCcCiNQ+TbV;>5ZJf8k559oX;ME'
    '5ZFVO!m71^at{KIw@aL<#XVq=0I#MUIk-pD{=~{Vz^Ijk`<0d4o~I;qW;ry_z~RUN;_e2J4$d&$Jq-$mu}~Ma7k0Ih)jp!sGv=gFOd4g8IJmGttJz*PQs%zY'
    '*f8J#xW@0|DYnjfq=xsnwF&S626LVc(y=`-r2azo6#j0KywI_@y(3{S3gIa2B3z;<udWYc>UL9$Q&KHMP|O7{=`Ty>(IBa(phP<VZ~n9Y;}1Iz9^l_U2Tu%V'
    'J=Pn&#RqJ^WeOP^AsR3YcR&W?^3fMh=`nh`y>E!dopea_VtzG4q?2I{Q=p>Ab2Ypd3HpA#BGqs-Zg_7|v0I4QFn=_sQ{?NWQ_(IxLz7+J#H}QIcnaTN&>i<K'
    '9hO=0<=0<@2QQu+9UUIQ;w+M}xV#}8Q*~IeX_`%7$-j%`E%l@%go7nUV+JbM=eVP+e&_?8Pf>=JU5TF|y>v%_EDijS<5Q)rHUvh<7PzQ*SgXB#^W^!<!#Aoe'
    'l|E`i*e5s6JHF{#g8MOGH5azu1&}wtN{5TF@YG=NbPY)J^8xzOWaQ&fs3Cndc+G^Zk?kYWf2^x8;h0O&o8~L|zOS4`=(-C;p=0uRA@;Jb?*2F=lWa^^&qzNB'
    'qztiUFld76at~Fgrw3yi!ZaY7R5Xwk>NUmahHy}!y|URVu!H=Y*eLp?g&dwFqSR>?0Wy)sW2hUqMdoA@o#M&I>^L*Gv<SA8Or=orpfAmN8c4hD6ogcYGeI#)'
    '8+(-NIk(#rUslvCiso|AWl>x~v<ZJHO*Ozd&(jdvja}`5Xn?x#9>YA~snm;Ia;YW=Dib=Tq+rBgH3S4KR&Pe*akcZnGsGYm{%h1$6_^)lcDljMiCznS1H?x3'
    'H1>;5<zkh=<s4_8-)ej>%|Su`C<4Pt6ldtVw8#*j4KJbf7YrL~o<xIps7?#&OBOVr$Frc&^a-*UVGi##Dxu^&4$2^-Jp<w*^@vKJpbkv5Zo#S`cNCNt;BCSQ'
    'g={_`K(Ad#7D|A-iA-Z%v}iTL+Yp*CI+5(1*rL7VEbRdumHH@-fS62X^DD6%C8PxdU_>+#FWJ@e-sBF=I3W}zy2$u7ZDFEVQ1Ufh^Nk9-5hGhhzDcW6MZ=d+'
    '>Z0Qy>&rBgOP+lt@(c&QCe8=suzD6x^W<De+L|tRHK$Y*L{T09G0J<^R%DwgGS(KWto9$E9?bY416UtU=UF<tdI4WWDMQP~x+p#Vl2S^)656R-k@|wpDa_If'
    '01+}h$2N5EWIOJcj)tiqmn%;Kb3K14$HASccYYSlVv>w(TG23<rT)W^c7s1~QhA|`Q)(?hvC(2WV3wshKx~6UoY1Xs0HC}&TnyOH2v0rt`C)a?oW#+zK)w)}'
    'XiXgS?*kt(7H~)^t|aI7G=}Md++M4RvPs#+3~xiq3x`at7)@a+W*g?<aOZrN%p~RwW((+Po|0a;&gIKb?aA419k3O;CsR*qXh@4|YZ8iPG@HfKVQ_70fxD#N'
    'FBV3(KcQmo$_XzCfhl?c8k*>?fK#QSwrhgmrvAMjpS-p+RP9xqgqjdX=cit;bfof-KK8y62ZqNwRYG2&H!P6wqUO&aJ%)mR=JhNsqbAC;a5i4#;b>Y^|4NOc'
    'c#!oi&11PYjpAaAxF<RUg!!zl)tb6Td8V<75rZ?^0v1NQk|#k&F%>Cq@*MclaPukvW8;m|Wp-7cs~8gzUd?Lhu3SvTwD4LTMZgV6$Bmp9<#rhs?ifs08LHuM'
    ';b;gD){T?8P-&SxkOFpA`f!X42h04Nqf@eAR2Sc1vrFq<^(dBE&2Bvk?ia-yVJrKY7!u~Rh{r?JhvdD$CR3UoH7xS7gs>RM&t3A2&)72Oeo5dc5krdw=Ot4S'
    't;#D5{y8@^4#brKL)MpC24Z5BOXEwN<SZJ*AIRrXd4d~$d1vu>*7GUwWuAKf+R>|*1@krPll-O4vd4k;oD6*Y<)1`(GQglC=}E`&G~D4=SjUQi2TFjflBl(w'
    '>!$8es<qAQPYYeYzpv#wzLhFY(HvcVN@-RpZqXPOJzv@!`KZ7a+CZDI92#k&bStf9d&TnLx5cEg`eVH)T#^otqc5H=AKMi6KkSWj%w-_BPE>Zs;T-6m;Xuh$'
    'k$X5K^^^BfRP`pe>oGMu1?&e9R*qtX&VIh^_-T^APtP(Z$*pI4Ko!Za#Y)()a-yn7o@T`W+>^Xpr`!dg9A1-cxY+9XwRdGtqKC1%QhSzMYWsaf2Z;U~C-FRL'
    'uSUhsu^6q*6WTOuwcDv~_p#M-=Qb_Jv%I3+cD%8_+;tg<$|RXq4y(1aHb;qU4C6s`RXu|3HJqoNtJ!HS@1C?*8>qc<BSU9@#eF&B4Q0W*8|6athqLeP&hqWE'
    'm~5ciIy&?@XTG%d{0x1b?o!i?Xp$S(c;Mc;Ch=Gfjd#gxgZv&U$PnYVw&bycrtO~TVaSnxOAq)6cDrgHf56!`TVMi|Vbmb2t?43vtEhuD{P&mfD<Zb0kY7X8'
    'TMo@~Ady~f?&JuHcvQs8XECr9$w2r92l*s@$BIR?p)Bu7ilMG~94Zl&BbUsHq*LLJ63<sqsqr|ypl)RN#o!Euz6w046xqd)F3}M>nUAlu08Q!n<_VrHB6Ll_'
    'B@(fg8;Di_cNDDB>{DR$&*3cyQWMnDsr3+Y*z=JmnNhTW%}{(K?~`cA9H_uN7SXckf|zN(0>g^DpsVCHr>&eQWR+&oEtG$U!p*->OsD88I?ou@s)M*_T(R(l'
    'IpEomZ5CS?rHkpX>s=cynQ@_3I2!B683MNFzd4RIZ#}0twLoh}8s4iT`a%*I4R`58OutKjq<2j(%&`;tk>BMtqgxXxADys~^z)OghWsSI@ix3`9}$i?c9u(S'
    'ieOK23|T=h;6TCys1itFq^pH}p&o-ACC9O3HgZ*Dba@2i@Z=XrBT2p`Z1N6XJVzfaQ!gtTL4%^~FnIJx^<7&Y8Kg0ja7~%`W3ka#z+>kE^nEVcgt-=*`-^qx'
    'sk?@Xq_uwn;OrQ5qaCYJEI9UUw=^Nq2(L6L_u29ud826&twJ<gmOLl$O)07%Jr!KrK=_a5^DJNjK@Qs@5rY`(TF7m>sfo$b9DT(;`P!C;zXWGOzE_&l@HOio'
    '81_(b)5f6LA-(4qxpvV`rpbhLhd4(ynu=ZvT?x-F6H0jQ1k`a|A`6``nHbGys!Mo;ZtvtOM%@u7!KHrM^E!_nZFOL;b?R%u6*`Ge;UK7luas{?A>aCjuu2Oa'
    'N_j#5>Zo7DnU{0$@DUYCzS;IA!?Z#3C`7LOc8gK~qrkF2u(TM8C=E!qha;C?3tYkq0?T`{3}}YPtK~wI(e#m33<ay`6I~2r@>S!^6oNglcM?smJb#5SCuurf'
    'YZh2A2GUwNew9Z6N2hg)_)BqozT%p0y-FhD!<C@IeYQ}A2Byu)IFpCV#!@j^;v~$ODmz|rj3P-0ql4w4O78T6@(QKIXI@)k6WSfA#0fn`NS+WJsfOMo4bOhi'
    'F=t^PgG$dMz4kn;kgIbMT0d?zymno?rBd7G#v(6xJmOWSup5<&xjikh;nZ(y*H;ZvLYO=CsM7}U-+3~|OvmUaENIsM-`=yuSaMu-cgco$uqDTi*}O=u$7FS9'
    'xt{InS9R}l6V_h86vsH;aX?thrCz-|WcJQ5ch);YyhI>j@ex^xlJG|qA*?7LfE4*BA`%D*Bg7ZN2^L?8A`u}=2oOp@0>U}f)!kLqeIGMklLg!B-ks@t>zsO='
    'XVp2UDsX)*MEt2OQQ9)Vtw-Zx-Ke6p>L^m%kJa<uH8bc<jI~K@^6&f#2LsJso!|9D^71EYIxYg*k$PfV-KbYU>#>;x;X6Y$B>-JqS84(%n$kYr&`XxppO>f%'
    '<IcimeZM)N+f{&a?;6nh0otrv*QQO;p9;||IaU`#S_=Jacd1$O#%DWcd>8l!le4FlyEDZDo4vzyP=CWZK!G)PWz*ri$@&HSbEWV2*3DU|GaVJaqxGeMkKe5d'
    'CiJgg0n%s}iLxlJ5@h<76}2^o15HEhXR`y<DhO4X7CtpPqD70NS$LeS^{#?b_W2INHm5sf)N5^_j-jLDDY)$z__RAO9tvTc?Ktc)p-eg~xjQCx*tcJA?Kb$<'
    '$B#dCvHIoGS!XJmC{u0Bff3zI#WiqFDl-BUDynQx_uE#6-kMr#NT*g)A2#tvPQ+ea5~u9$#YZj-9GO-|R5dxb1J4)O+8wo*a_E65J1}y(<9^3hR-=~sa=%-R'
    'b!5*=^Od!xa!$w0YQN;{!g)YFd($UE>+R8EQ^0KnN_6Dt?W4_HSM}uu)$Xh?>`%E<d86XEnzp_9bIATdVAF{@Kba`75hj-(e`>JZFw4gHD%&4fQV7h3(gGRT'
    '=7?K{rfFVdvCGgbb&%T?jo1z_r~|=GctKg*OSvVI^{auui!dX7z>C=*x{>+aWe=+W{9&zyhi>-3QR|g_@#A0kr58{1dicD@O*QfE5va0GW?T$XT~_X-pOk@%'
    '^%dxesbqH8v=b~9u(ipTzS3iUf=9=qtju2b)&&?90o;u2juvo_HtNrpPwXhOV478|yWP-DO{T+!uzc>Z%dFUYWgXkM&W6>!cG6m9N|akv_|evC?I`J;T<s6f'
    '%|+FOL*mAcs)%b?PRD4gOmZqcYioC}-X*hZ6*r&SC#^d7#lO*_3?8~Z8QWE6IxdK5Kz0?!hlYlF*G=J!3S_w@%n!ivAFQpdUs3yV^RmL2!7hWnn##7pJgd6h'
    'yw-=WbB9^TyHf8PHwaC!kSb2)Q#aO}pk!SmQtu%+$J)bw_ca3T;tmy3jR&FWq&?U2(XS6t?Ey&1)RHW7tFc09Q}Y3#-SDi*O1zi&mJ#VSy>$w6SKPmWvi<Ij'
    'SO04D-Mt<@_qej%x>2B#>rI+qBh<D0ehT>Oe7S&@oxi1?1?~d2`xt;(S=qP(uEr*MIa7r;5{3s)_I3~YC86{;Y9)~MO@f&QhRR`kck?n{tWtJirTVyZ;`-Mn'
    'r6(oV`=4kIMs-^}xG>@DZ0kBZn4+cJ-`Ub?YL`u>K!sPQ>npD_%dt{F%T>s7RFRdFIH@lyY9hq(Q0L*5mF2CDo0(F0<lMm{su&!sUYl{Fb$n3vT@)4A!Bgj-'
    'xN!dINAL#I@yE_T^2B2sPhWUw*tcI}(&o=Ud2z`A=W;mqeHn(6*Nhr8cXVh<H`xkR^2>&rOz~W<j1c$V>e{Af%k0CB&iHvTWkB;<oe6XCI@{4XiW(=<7f#!x'
    '-c~E)C#M)BfG4#~;3iC8fY#bk=<O+1XTo63kTSue%7%-Z${qDP?e1grxJk*-mAc?p6KX#<Y+JLwDMx^e|G<Nf{z54nbeC}$8VnvRmwj1v>WWn;f3_bkfoGyH'
    '3xHu169cy2yS{D4H?LlSUJkv}uYdA(7-r%szuelyz)_Wet1@Fa*kON+@d3C#ph7pLHWtj(>r|F9Hl5)CQCalFWKnJ6wiLfsM&n|fQ`ucZL~HA1@a&+DV8rd1'
    '8vbN&Q2u<n9P;$ncJ~HkJ?P($sGK%s-%KO(%e+=XVaDKi-?S)ui_B6Z9rn>;DHAZ9HV2(;s>$&i?Gsa@4R7bEklyc%^vs#Q3F0|ufBB&PvKdwL?fnqH4LZ^E'
    'w+8<+x{go{AIr|!NUCO4mupOQJA=w=h_Ms3`Gw=g7Xw2dWr)tm&_wy0iP7KO*Cv(uNNfSq46KiyIXmpb{|DskK{$l@I=dfUs=xjW;%I+fnJ%bA5q7z}gMB;*'
    '+$We#IXJ+d4{!kenu^MCnfqrdg*@s%ybfI&U)kOpRB20gZh}q$D+6)N;P96^{AF*V_XT_yy5pCYhh*$9JtM<rXs4Tg!s?X6tUknHnhv-ky-8))nr>*4OomOO'
    'U{m$F-eoYc(5Gjp7f0y#tt!^$U32?@lS=Mx;+jEv3UXmD*IJ3_O=Q~a#KiISN+whvxEYGeC6^y9lkjL+gq7y&qDWMs^XR8T=X}_lku(fDXrvQpBzqA%vL}aD'
    'e>!Z1CdaQ7<tv9vd|xtavLj>C^vb+AbcW<%vm=eVY-L8y%zZcgYej4=PD)%VA+hey)bCAK@_H+2y*ppMv4&Q<G8;>kZl$Zw$X7QRDOx1Y&DHc+v_?^_?PXBh'
    'Y8NxUYLKWVP?}tFb0aP8>Bw<EN{*hEFn#TQb>-;Y*RKy`nGn}@nK(c}SN7ZlNMi+3CGZxfYPs+thUs>(R|y0sS$<}Y{H)ygYTjBY@?F_zH|A}7seKskcwt%('
    '#&_yiOl!x)ZrT;OZ(oDZ87N-GnlA&$y*8mcayzlW4UK1ARyT1l%!$)%Zvxg(7VZSj*b1CmUljKftmMAd_Fq}ssNSmZ4i*=0ESj8%%E*^xGKM)NHI#{6C@Zkc'
    'b29V{uG<$*YLBD+`ewTyi+f{{#+nuA(Vz7t>>*jAhjtSy3`GZ9T(WAfw{}_Em(5yRvK@DS;^jeYH@~|{eJ`ljsNOPYnmMa7>R77QSM3pxX3k%$+5D~G)|T?T'
    '4dK>!=fkcS(Z^Dzzb?81^v%3G-9d-`>zdYP1?aEZ)MP^cJYuolI{V<WAkJF*H+<vVF#W7;b5rZtzE#HAMt3WS9oFqyYq?0T$CF#UL0H}x+>#QD8`>8O&yXC8'
    '&lNiMByX_;<9|8DPd<NpuZPbEIu8{(PmMbcioL_(ykB3|T<PDVofhB3?fW<Bxo~@1zFgU=jn9F%>NfscX8n)Ctgl}#Ho|LJP&2%5<-4c+lkMi9|1Ep`aQ66p'
    '`V1CK28;zY<d3(A?%)+eXLLHyIb%3zsca6HG?QUYr8(B!I#@qyh(+7@?`rk`#0xl7<j7Lq!jgd_CU_Uvc`2xkpHRc^_7Iwgfj7)kS<r*Ge3V<>N0~);e&7CA'
    '{TE-$EF1sxmg91B9hX}qEb90F=zHJX0`CQ%_j(?TCt}*(ODiJk`V}<oKdIDCU0t`X)NaKtVT6<_i;dy;OmTh_=>+e#woqeC?OQpAb|HweI)okhv{IKw!{{o#'
    'RawVd^-Hu<T|Yu=b1R?-BTMS&s6z)w9WoZT34nTbv%E~!d|HQREehAwQ#HQ0aem|C&{%(ms>}W+iLL@>=&+GM*@P>dW)OTrogA4C4z}-1cT3%$K8>eom#0nW'
    '(vzSl)h^cWwds7i1LDKZ9zzmn*8W^o@Tfjn0`t;}V5-#KDk{Me<J^lpEWRv<fQ1nDcDJ{tE)&!}sESMK>8|8(dFfyIvsFeHGjG~t*O)IeC(8_?y3n$oJ9B3w'
    'o8Ya;Xct(d{!BM95A0v+%aP5FL_I1j9c&e=r~@H!ezvbZxVybGh(e?zsS8oW^^cmcN2|N4KxwmTC8D&TJi5KR`Rg#bZ0q5z&7HyQubKrJ7%OFT6o0=qWk#K}'
    '+>Lx=yNt@4(YGh62_E1QjOuk5!DCv~_e_-wW{kBehg%=kYuavqJg)0<krbm1EH|^gTvb`uQn(E!<15=+YC0lAQ?YBb0KS`_3<6qPNj6Qd*Vjm|TBPr@(#F9~'
    'a|fci(TLfq3R$Px2hfU;&E=#D>$**t)(T+VZ@lKGJtI!s&r;0!dh-_!buMYk8M^@K6ir4Qn}o)@e>Q2dO!d8&H{mE~j(7bd@uufEeAdX1>X}u(^t^Tp9Xb!('
    '8}Gy6)$)!nGsg+iSE5-Y(ve&4f^usGYTC7FU@?F$A5z2VG9(TKt#LhD@EW1L=$0j$%jTawp>A*1!@Y0m|K=&ng~9KBn}L6V&nmpBJg2eF0X1ZDl}$DakP5mI'
    'F&1z(L^wTDU{vIpYMXXEy)4IDy8)re___>&Ab}aE_H%R<<FjCCB`V>mB19@8L{LO=rbHHyOz<KXRHQ0VVOFF?l1Fh&X&Q(ur=%!E$bu{`HyLJipzQ)yn*@vm'
    '<1h$rg>LAng=q=`Qg?^*LPa4DGnG+U<b@(K42w7t2}$ETh)Kp$7DqG?g6CPBr!*`A=t!DHB$iP!r#oG}A+ZjBoObplVM#z43-X*w8cHT;5r%mt7$sb=SQ3?|'
    'LZ(F?r$I!MRKoWQM)Q=3K!tI5)V?&-)2vcJBL%abB#8(T8HhLoWI;{^fzCxdpt*t{MaH676jTIh5eF%}i*mx#I4-+V)xc?PHUrQ_7{moh2!RG^AfuRsGz_vV'
    'i(_aXDo%qy1))#`s3a9UPI)Xi7Xo@i2o@hg(50hw=Hf%+^Lu-EZ}RrUz-KaxQw5VKvQV*{WO=A$NO=OYlnEiB@0`*i5-ewoaVlVqdB9So6e}nT9QZ~!7|<%s'
    'fJoyQ=p&0_nT15aJR~VELdtSj2O{$nDJqUR0osj70=<qHkT#<c6%Ode5jN0yL9#?-JQPu^0Gi09$a$VX2MM8E1%jl&E-8(oT=FD^zEhsT9D#O;r63*98{5U;'
    'h-AjIC<|klpn^qF$f7hzqO?dk08ew0#Bq`dm1QbUk|-|1Br8PBVdhvGC0!$^j5~uNNb)e2DT_oJGM4Znfptz)N};EmQxyxCd>Y6?s4yoiBoPnuGz(HG11SWZ'
    'gTXm>#^s<9E5azzv?n-=WXwpKL<Q1O45LsaVKf%N%|!%ABq`D;6$SJ_5SR%U)%Mk^H4Q}}NkJ45DU+ngIZYVQG)<ALA}Kjd6QEj9D=BbdkcLs3#ZeeUii;#o'
    '66FrU7|BxuJ5vdWyc`%BXpS<Vj}TT=aiEo$5|L%8ih0OYktG=`K=^P4EWk6$fDK5@9N5k#hCu*hPkF+*l01xnwZbBmz_d~&4AwkIfp{fIWC4>BBs72}4kUx+'
    'P#nO+8wbIz0^stR*ev%6tC^QfrJ(>JK^e~&0TKleOhqIE0T2R-hJ_#?SVh32j8d9$M#BskIRs%wa|cB&nhfv+2r$ios56jyDFrDYa>k1w0B|{v0}A9AsVt0&'
    'jB=667z9-g<Qy;{UO>|g+$Wv@Xgm#Ju8SfAiitteDiG)j#9F8#s)7PAP=T_7D3Hko&~F$?5~!R2Rmc)cS@3;&hF!kAizV*W`10<i98VRNQm@v76wrIZQVzVv'
    'a^NBff)ANj25~NBM$#Z-oF_rTf$t*;%$CC}2C2$8h*}p@r%Y7|EN2vvD9^K?;5<x0#Y&n+F-#Bv5tkGo-va1)KqB}hR3Hgcpl1@tpv^oKX!U#&Mg%raSDgT@'
    '#N!zBAdxgH7>ut_0gv-SiXtgsJQRpb<SEQ7@FwWaFp;682~03y5hz$P8()0}g%cO$Fjb0zj7S2IfAE)NISeR-Ss@Cw2XGG96Tp#K6j9(%e64tzatf+r0hUG+'
    'qJ3f*KPzBZz`~h|a^Or{G*BBHER;|c66g+8Z30RU=%2#aphq;$^O%Yvj!7KNMCsUfl_unQn59C1dBal>iZO{o5GDwk3z$t1TkwZ~0mKMZ5+qO<2$}^#MFEJP'
    '!bSEhNc!TS+UCAii83H7(Bx=Yfo@4DqcDY01iS-40wUmtFbCTvgm)Q{XzR$B$5}4G+DSlLlt{b_N|AqP!i5A;pGDvyBowq_p72Dbpm13MoFt?I5(K+SWQt25'
    'OqmpcND5Gt62w9aZkx)$qyb^6l0wJ;I42}B2Ac<jCnr=A5rEn#0w9hQ>5zidK=J}JErpp(z*cLaaLEhwGYFPgCOpa`&_+=hff>Ux)OZAh71#|R#WLW~Afqsv'
    'JOb&N<V;3E1PFs!;Qe8chy)z05Kw<1&~rd#7=ekBMunuDgCGWK6=?z!Clj#ofT*?h0p_RBE-c!3yh98mr5LQ8lz_;jTGNRv389NHDKLRxL}W!0<}yQ>EJD!c'
    'z@{LuL!NS($|8zZS8jlVa_soKeynj(e&vUM`-QuX^?LZwUi04{{3iZ=L{DTMl97Yt^0Apnmh(1zR9}DXj(g7oP<$S8P`z^eh^Txk-beY`%MbkBM*tl@4>&0I'
    '-!=o~p}Ynk-{(K@!o7C_E_}{A_)fn4DEK_jz(@BFx4nAT*Wo98-szxw{>OX0mZtKXfB*64A3O2Wy&gXAa+*)x(d%`Y$v%WHf9A74_vFc551;os5MCU55FAt3'
    'hx5vRK5^>XK;!tl+ktcc1rLs8?)p%k`^w%&!>rfC=UxZOgU`)?V#^92-1lBS_qFJQy&gVyI^bS<sR37+t$yp@_}t{+ZTIwg_}t~RzUP&?wPl+6?LTqycYgcP'
    'J9<5QPCD%$`^UPyZ+`kf)F1xWZ+#7z3!nEmKwkcS4Ww;i`e45Jj&nOt1H<6+lMa|a{*M}r=L-5T{`xQ8xB9uedp&$yTFw5uhS9ite)BtD`p^sCdVjBn&)rV*'
    'e}Ct(UWX;?132-Ew|(RHR(d^r-tPeT;d&duJi0jjk=Oq82e<cn`1Bn}-^!2m7Fnb|v{Qfi=^uPI==Jb%m*^)RX`n6S<-h;e6W`ni*1_kL1Li~f4Vd|sr4RG|'
    '7ryk}7XcnV?v#D!v)wQ&_3lIZ#@Xlpsp{RD1L@DcItP+(A^KqN*tznh0JsyM`y8;J{_{?-z7^;LeDqI#>eB7Nt@ymp0r=0~X#jRudOjfkg82qe@^8ny>*ibB'
    '=2Z-9+J5=e@!9pmd@z1-ng+~s!{gnhYJ3oWD0&0pZ!R2P5SZ@6@`FMeSfBs+@i`$PJ{*7Jwt+)``S=psxKDoc#M|-vAA~=}ZN1*BzY4Ok_dlt%QqT'
)
EMBEDDED_SOURCE_SHA256 = '976c9eded41b4f4034f6a88d1a770649ce331e8621a070f4eec7e4412b568eb7'
PROJECT = WORKING_ROOT / 'TickNets_LLargeV2Adam'

def project_prefixes(archive):
    names = {info.filename.replace('\\', '/') for info in archive.infolist() if not info.is_dir()}
    prefixes = []
    for name in names:
        if name == 'train_cifar10_large_v2.py' or name.endswith('/train_cifar10_large_v2.py'):
            prefix = name[:-len('train_cifar10_large_v2.py')]
            if all(prefix + relative in names for relative in SOURCE_PATHS):
                prefixes.append(prefix)
    return sorted(set(prefixes))

def find_project_source():
    if SOURCE_ZIP is not None and SOURCE_ROOT is not None:
        raise ValueError('Set only one of SOURCE_ZIP or SOURCE_ROOT')
    if SOURCE_ZIP is not None:
        candidate = Path(SOURCE_ZIP)
        if not candidate.is_file():
            raise FileNotFoundError(f'Source ZIP not found: {candidate}')
        with zipfile.ZipFile(candidate) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError(f'Source ZIP must contain one complete project; found {prefixes}')
        return 'zip', candidate, prefixes[0]
    if SOURCE_ROOT is not None:
        candidate = Path(SOURCE_ROOT)
        if not candidate.is_dir() or not all((candidate / relative).is_file() for relative in SOURCE_PATHS):
            raise FileNotFoundError(f'Source folder is incomplete: {candidate}')
        return 'directory', candidate, ''
    zip_candidates = []
    directory_candidates = []
    visited = set()
    for current, directories, files in os.walk(KAGGLE_INPUT, followlinks=True):
        resolved = Path(current).resolve()
        if resolved in visited:
            directories[:] = []
            continue
        visited.add(resolved)
        if 'train_cifar10_large_v2.py' in files and all(
                (Path(current) / relative).is_file() for relative in SOURCE_PATHS):
            directory_candidates.append(Path(current))
            directories[:] = []
            continue
        for filename in files:
            if not filename.lower().endswith('.zip'):
                continue
            candidate = Path(current) / filename
            try:
                with zipfile.ZipFile(candidate) as archive:
                    for prefix in project_prefixes(archive):
                        zip_candidates.append((candidate, prefix))
            except zipfile.BadZipFile:
                continue
    if len(zip_candidates) == 1:
        candidate, prefix = zip_candidates[0]
        return 'zip', candidate, prefix
    if not zip_candidates and len(directory_candidates) == 1:
        return 'directory', directory_candidates[0], ''
    if not zip_candidates and not directory_candidates:
        import base64, zlib
        payload = zlib.decompress(base64.b85decode(EMBEDDED_SOURCE_B85))
        if hashlib.sha256(payload).hexdigest() != EMBEDDED_SOURCE_SHA256:
            raise ValueError('Embedded Adam source checksum differs')
        embedded_path = WORKING_ROOT / 'TickNets_LLargeV2_Adam_Embedded_Source.zip'
        embedded_path.parent.mkdir(parents=True, exist_ok=True)
        embedded_path.write_bytes(payload)
        with zipfile.ZipFile(embedded_path) as archive:
            prefixes = project_prefixes(archive)
        if len(prefixes) != 1:
            raise ValueError('Embedded Adam source is incomplete')
        return 'zip', embedded_path, prefixes[0]
    attached = sorted(path.name for path in Path(KAGGLE_INPUT).iterdir()) if Path(KAGGLE_INPUT).is_dir() else []
    raise RuntimeError('Could not identify one complete TickNets source in Kaggle Input. '
                       f'ZIP matches: {zip_candidates}; folder matches: {directory_candidates}; '
                       f'Input entries: {attached}. Attach the source dataset, or set '
                       'SOURCE_ZIP / SOURCE_ROOT to its exact path.')

source_type, source_path, prefix = find_project_source()
if source_type == 'zip':
    with zipfile.ZipFile(source_path) as archive:
        members = {}
        for info in archive.infolist():
            if info.is_dir():
                continue
            normalized = info.filename.replace('\\', '/')
            if normalized in members:
                raise ValueError('Source ZIP contains duplicate filenames')
            members[normalized] = info.filename
        manifest_name = prefix + 'large_v2_adam_source_manifest.json'
        manifest = json.loads(archive.read(members[manifest_name])) if manifest_name in members else None
        payloads = {relative: archive.read(members[prefix + relative]) for relative in SOURCE_PATHS}
else:
    manifest_path = source_path / 'large_v2_adam_source_manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else None
    payloads = {relative: (source_path / relative).read_bytes() for relative in SOURCE_PATHS}
if manifest is not None and set(manifest) != set(SOURCE_PATHS):
    raise ValueError('Source manifest does not match the required files')
PROJECT.mkdir(parents=True, exist_ok=True)
for relative, payload in payloads.items():
    parts = PurePosixPath(relative).parts
    if not parts or any(part in ('', '.', '..') for part in parts):
        raise ValueError(f'Unsafe source path: {relative}')
    if manifest is not None and hashlib.sha256(payload).hexdigest() != manifest[relative]:
        raise ValueError(f'Source checksum differs: {relative}')
    target = (PROJECT / Path(*parts)).resolve()
    if not target.is_relative_to(PROJECT.resolve()):
        raise ValueError(f'Source path escapes project: {relative}')
    if target.exists() and target.read_bytes() != payload:
        raise RuntimeError(f'Existing extracted source differs: {target}; use a fresh session')
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
missing = [module for module in ('torch', 'torchvision', 'numpy', 'PIL', 'pandas')
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(f'Missing Kaggle packages: {missing}')
import torch, torchvision
assert hasattr(torchvision.transforms, 'RandAugment'), 'torchvision RandAugment is unavailable'
assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator before running.'
print('Project source:', source_type, source_path)
print('TickNet-L Large v2 Adam source:', hashlib.sha256((PROJECT / 'train_cifar10_large_v2.py').read_bytes()).hexdigest())
print('Runtime:', torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))


Project source: zip /kaggle/working/TickNets_LLargeV2_Adam_Embedded_Source.zip
TickNet-L Large v2 Adam source: 2428e5e5fcd060a21d46da48eb17d1c0968f021361d78758c41a3cb967415849
Runtime: 2.11.0+cu128 0.26.0+cu128 Tesla T4


In [3]:
import subprocess
command = [sys.executable, 'scripts/run_kaggle_large_v2_adam.py',
           '--dataset', 'cifar100',
           '--cifar10-source-root', str(CIFAR10_SOURCE_ROOT)]
if EPOCHS_PER_SESSION is not None:
    command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
if RESUME_ROOT is not None:
    command += ['--resume-root', str(RESUME_ROOT)]
subprocess.run(command, cwd=PROJECT, check=True)


=== CIFAR Automated Downloader ===
Destination: /kaggle/working/data
Target datasets: ['cifar100']

[*] Attempting download from: https://cave.cs.toronto.edu/kriz/cifar-100-python.tar.gz
    -> Downloading: 161.2 MB / 161.2 MB (100.0%)
[+] Successfully downloaded and verified: cifar-100-python.tar.gz
[*] Extracting cifar-100-python.tar.gz to /kaggle/working/data...
[+] Extracted: cifar-100-python.tar.gz

[✓] All target CIFAR datasets are verified and ready for training!
Epoch 001/200 LR=0.001 Train=9.09% Val=16.62%
Epoch 002/200 LR=0.000999938 Train=17.78% Val=28.20%
Epoch 003/200 LR=0.000999753 Train=25.13% Val=36.14%
Epoch 004/200 LR=0.000999445 Train=29.26% Val=39.12%
Epoch 005/200 LR=0.000999013 Train=31.69% Val=43.02%
Epoch 006/200 LR=0.000998459 Train=35.03% Val=45.46%
Epoch 007/200 LR=0.000997781 Train=37.71% Val=48.34%
Epoch 008/200 LR=0.00099698 Train=38.94% Val=51.22%
Epoch 009/200 LR=0.000996057 Train=40.98% Val=52.22%
Epoch 010/200 LR=0.000995012 Train=42.17% Val=54.04%
Epo

CompletedProcess(args=['/usr/bin/python3', 'scripts/run_kaggle_large_v2_adam.py', '--dataset', 'cifar100', '--cifar10-source-root', '/kaggle/input'], returncode=0)